# Phase 6 -- Abstractive legal summarisation: all runs in one notebook

**The problem in three facts.** (1) Judgments are thousands of words; every Phase-1 model reads at most ~1,024
tokens. (2) Summaries are long too, so one forward pass cannot write them. (3) The metric, BERTScore F1 with
roberta-large, reads only the first ~510 tokens of each summary -- the opening of the summary decides the score.

**The approach: plan, then write section by section** (same for every run, so runs differ only in model + technique)
1. **Segment** each judgment into ~450-word parts (sentence boundaries).
2. **Align** every sentence of the reference summary to the part that supports it best (IDF-weighted word overlap,
   near-monotone dynamic programme). One long (judgment, summary) pair becomes many short (part, sub-summary) pairs
   that fit the models -- divide-and-conquer training (Gidiotis & Tsoumakas 2020).
3. **Plan** the length: total summary words from a power law of judgment length, shared out over the parts by a
   learned **positional profile** (which parts of a judgment summaries draw from). Parts with < 12 words of budget
   are skipped -- content selection for free.
4. **Write** each planned part with the adapted Phase-1 model, conditioned on a cue
   `[<position> part] Summary (about N words):` (length + position control, Fan et al. 2018); join in order.
5. **Tune** one knob on held-out documents: the length scale (0.8 / 1.0 / 1.25 x plan), chosen by BERTScore.

| Run | Phase-1 model | Technique | Why this pairing | Submit |
|---|---|---|---|---|
| **S1** | D (encoder-decoder) | **LoRMA** r16, every attention + MLP layer (encoder, decoder, cross-attention) | D is a true seq2seq model -- the natural summariser; best backbone so far; compulsory method; KD teacher | yes |
| **S2** | C (prefix-LM) | **Houlsby adapters** (bottleneck 64) | C reads the part bidirectionally and writes causally -- exactly how it was pretrained; adapters were C's best method in Phase 5 | yes |
| **S3** | B (local/global decoder) | **QLoRA** (frozen 4-bit NF4 B + LoRA r16) | 4-bit base = half the memory at generation time, the cost that dominates summarisation | yes |
| **S4** | A (Llama-style decoder, 163M) | **LoRA** r16 | the standard low-rank baseline on the largest decoder; prefix tuning failed on A in Phase 5 | yes |
| **S5** | D (encoder-decoder) | **Prefix tuning** (32 learned states at every encoder layer, from real activations) | prefix tuning was introduced for summarisation with an encoder-decoder (Li & Liang 2021); same backbone as S1 = controlled comparison | yes |
| **S6** | half of D (layers 0,2,..,10 of encoder AND decoder) | **Knowledge distillation** from S1: gold words + the teacher's word distributions (T = 2) | smaller, faster summariser; Kim & Rush (2016) word-level KD | yes |
| **S7** | -- | **Consensus**: for every planned part, keep the version that agrees most with the other systems (segment-level MBR, ROUGE-1) | system combination without any extra model; built here | yes |
| Lead / Extract | none | first N words / most central sentences of every planned part | does abstraction beat copying? (dev only) | no |

**How to run** -- inputs: `legallm-tokenizers` (`t1_bpe.json`, `t2_unigram.model`) + the four Phase-1 checkpoint
datasets (A, B, C, D). Secret `HF`. **GPU T4 x2, internet on** (internet also fetches `bert-score` / roberta-large for
LOCAL scoring only -- they are never used to train or to write summaries). **Save & Run All**: about 5-6 h.
Submit `p6_out/S1 ... S7/submission.zip`; download `p6_out/report_files.zip`.

## 1. Inputs: tokenizers and checkpoints

In [ ]:
import csv
import gc
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import torch

NB_T0 = time.time()
N_GPUS = torch.cuda.device_count()
print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "p6_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = WORK_DIR / "ckpts"
CKPT_DIR.mkdir(exist_ok=True)


def step_of(p):
    return int(Path(p).name.split("_step")[1].split("_")[0].split(".")[0])


def _rebuild_from_folder(folder, dest):
    """Kaggle unzips an uploaded checkpoint (a .pt file is a zip archive inside),
    leaving a folder with data.pkl, data/0, version, ... Re-pack it into a .pt."""
    pkl = next(Path(folder).rglob("data.pkl"))
    root = pkl.parent
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_STORED) as zf:
        for f in sorted(root.rglob("*")):
            if f.is_file():
                zf.write(f, arcname="archive/" + f.relative_to(root).as_posix())
    return dest


def _collect(prefix):
    """{name: path} of usable <prefix>_step*.pt checkpoints under /kaggle/input:
    a .pt file; a .zip holding the .pt; the .pt renamed .zip; or a folder that
    Kaggle unzipped from an uploaded checkpoint (re-packed here)."""
    found = {}
    if not KAGGLE_INPUT.exists():
        return found
    for p in KAGGLE_INPUT.rglob(f"{prefix}_step*"):
        stem = p.name[:-3] if p.name.endswith(".pt") else (p.name[:-4] if p.name.endswith(".zip") else p.name)
        dest = CKPT_DIR / (stem + ".pt")
        if p.is_file() and p.suffix == ".pt":
            found.setdefault(p.name, p)
        elif p.is_dir() and any(p.rglob("data.pkl")) and dest.name not in found:
            found[dest.name] = _rebuild_from_folder(p, dest)
            print(f"  re-packed unzipped checkpoint folder {p} -> {dest.name}")
    for z in KAGGLE_INPUT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                inner = [n for n in names if Path(n).name.startswith(f"{prefix}_step") and n.endswith(".pt")]
                for n in inner:
                    dest = CKPT_DIR / Path(n).name
                    if dest.name not in found:
                        with zf.open(n) as src, open(dest, "wb") as dst:
                            shutil.copyfileobj(src, dst)
                        found[dest.name] = dest
                if not inner and z.name.startswith(f"{prefix}_step") and any(n.endswith("data.pkl") for n in names):
                    dest = CKPT_DIR / (z.stem + ".pt")
                    if dest.name not in found:
                        shutil.copy(z, dest)
                        found[dest.name] = dest
        except zipfile.BadZipFile:
            continue
    return found




def checkpoints(prefix):
    found = _collect(prefix)
    assert found, f"No {prefix}_step* checkpoint (.pt, .zip or unzipped folder) under /kaggle/input -- attach it."
    return max(found.values(), key=step_of)



def find_one(name, label):
    hits = sorted(KAGGLE_INPUT.rglob(name)) if KAGGLE_INPUT.exists() else []
    if not hits:                                   # renamed copies, e.g. "t1_bpe (1).json"
        stem, suf = name.rsplit(".", 1)
        hits = sorted(KAGGLE_INPUT.rglob(f"{stem}*.{suf}")) if KAGGLE_INPUT.exists() else []
    if hits:
        return hits[0]
    for z in KAGGLE_INPUT.rglob("*.zip"):          # the file may sit inside a zip
        try:
            with zipfile.ZipFile(z) as zf:
                inner = [n for n in zf.namelist() if Path(n).name == name]
                if inner:
                    dest = WORK_DIR / name
                    with zf.open(inner[0]) as src, open(dest, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    return dest
        except zipfile.BadZipFile:
            continue
    seen = sorted(str(p) for p in KAGGLE_INPUT.rglob("*") if p.suffix in (".json", ".model", ".zip") and "_step" not in p.name)[:40]
    raise AssertionError(f"{name} not found under /kaggle/input -- attach {label}.\nFiles that might be tokenizers: {seen}")


T1_PATH = find_one("t1_bpe.json", "legallm-tokenizers")
T2_PATH = find_one("t2_unigram.model", "legallm-tokenizers")
CKPT = {k: str(checkpoints(k)) for k in ("A", "B", "C", "D")}
for k, p in CKPT.items():
    print(f"{k}: {Path(p).name}")
# local scoring only (never used for training or writing): bert-score with roberta-large, as the evaluator
try:
    import bert_score  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=False)

## 2. Code files (written to /kaggle/working)

In [ ]:
%%writefile /kaggle/working/legal_d.py
"""legal_d.py -- Model D: encoder-decoder legal language model + its trainer.

Model D (T5 / RedLLM-style encoder-decoder, tokenizer T2):
  * Encoder: bidirectional self-attention over the prompt.
  * Decoder: causal self-attention over the continuation + cross-attention
    to the encoder output.
  * Shared (tied) embedding for encoder input, decoder input and output head.
  * RMSNorm (pre-norm), RoPE in self-attention, QK-norm, GQA, SwiGLU, no biases.

Trainer: one process per GPU (DDP via torch.distributed.run), Muon optimizer
for the hidden weight matrices + AdamW for embeddings/norms, fp16 autocast
with GradScaler (T4 has no bf16), time-based warmup-stable-decay LR
schedule (so the LR always finishes its decay when the time budget ends),
flat checkpoints with resume, periodic push to a Kaggle dataset.

Run training:   python -m torch.distributed.run --standalone --nproc_per_node=2 legal_d.py d_config.json
Import helpers: import legal_d  (model, generation, scoring)
"""
import contextlib
import datetime
import itertools
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

ARCH_ID = "D"

MODEL_D_CONFIG = dict(
    vocab_size=32_000,        # T2 (SentencePiece Unigram), same as C
    d_model=768,
    n_enc_layer=12,
    n_dec_layer=12,
    n_head=12,
    n_kv_head=4,              # GQA 3:1 (self- and cross-attention)
    d_head=64,
    d_ff=2048,                # SwiGLU (same FFN parameter count as C's 4x GELU)
    max_enc_len=1024,
    max_dec_len=1024,
    qk_norm=True,
    mlp="swiglu",
    rope_theta=10_000.0,
    tokenizer_id="T2",
    arch="encoder-decoder",
    tied_embeddings=True,
)

# ---------------------------------------------------------------------------
# Tokenizer (T2)
# ---------------------------------------------------------------------------
sp = None
EOS_ID = None
DEC_START_ID = None
BANNED_IN_OUTPUT = []


def load_tokenizer(path):
    global sp, EOS_ID, DEC_START_ID, BANNED_IN_OUTPUT
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor()
    sp.load(str(path))
    EOS_ID = sp.eos_id()
    assert EOS_ID >= 0, "T2 tokenizer has no <eos>"
    DEC_START_ID = sp.bos_id() if sp.bos_id() >= 0 else EOS_ID      # decoder start token
    BANNED_IN_OUTPUT = sorted({sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()} - {-1})
    return sp


def encode(text):
    return sp.encode(text, out_type=int)


def decode(ids):
    return sp.decode(ids)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(x.dtype)


def rope_cache(n, d_head, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(n).float(), inv_freq)
    return freqs.cos(), freqs.sin()                      # [n, d_head/2]


def apply_rope(x, cos, sin):
    """x [B, H, T, d]; cos/sin [T, d/2]. Rotate-half form, computed in fp32."""
    cos, sin = cos[None, None].float(), sin[None, None].float()
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half].float(), x[..., half:].float()
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


def expand_kv(t, n_rep):
    if n_rep == 1:
        return t
    B, H, T, D = t.shape
    return t[:, :, None].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class KVCache:
    """Decoder self-attention cache (allocated on first use)."""

    def __init__(self, max_len):
        self.max_len, self.pos, self.k, self.v = max_len, 0, None, None

    def update(self, k, v):
        if self.k is None:
            B, H, _, D = k.shape
            self.k = torch.zeros(B, H, self.max_len, D, device=k.device, dtype=k.dtype)
            self.v = torch.zeros_like(self.k)
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class SelfAttention(nn.Module):
    def __init__(self, cfg, causal):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.causal, self.n_head, self.n_kv_head, self.d_head = causal, h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, key_mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        first_chunk = cache is None or cache.pos == 0
        if cache is not None:
            assert T == 1 or cache.pos == 0, "cached decoding feeds one token at a time"
            k, v = cache.update(k, v)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(k, rep), expand_kv(v, rep)
        if key_mask is not None:                                  # [B, Tk] True = real token
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask[:, None, None, :])
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and T > 1 and first_chunk)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class CrossAttention(nn.Module):
    """Decoder queries attend to the encoder output (no RoPE: T5-style)."""

    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def project_kv(self, enc_out):
        B, S, _ = enc_out.shape
        k = self.k_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.k_norm is not None:
            k = self.k_norm(k)
        return k, v

    def forward(self, x, enc_kv, enc_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q = self.q_norm(q)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(enc_kv[0], rep), expand_kv(enc_kv[1], rep)
        mask = enc_mask[:, None, None, :] if enc_mask is not None else None
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg["d_model"], cfg["d_ff"]
        self.gate_proj = nn.Linear(d, f, bias=False)
        self.up_proj = nn.Linear(d, f, bias=False)
        self.down_proj = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = SelfAttention(cfg, causal=False)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, key_mask=None):
        x = x + self.attn(self.attn_norm(x), cos, sin, key_mask)
        return x + self.mlp(self.mlp_norm(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_norm = RMSNorm(cfg["d_model"])
        self.self_attn = SelfAttention(cfg, causal=True)
        self.cross_norm = RMSNorm(cfg["d_model"])
        self.cross_attn = CrossAttention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, enc_out=None, enc_mask=None, enc_kv=None, cache=None):
        if enc_kv is None:
            enc_kv = self.cross_attn.project_kv(enc_out)
        x = x + self.self_attn(self.self_norm(x), cos, sin, cache=cache)
        x = x + self.cross_attn(self.cross_norm(x), enc_kv, enc_mask)
        return x + self.mlp(self.mlp_norm(x))


def _ce_chunk(h, w, t):
    logits = F.linear(h, w).float()
    lse = torch.logsumexp(logits, dim=-1)
    valid = (t != -100).float()
    nll = (lse - logits.gather(-1, t.clamp(min=0)[:, None]).squeeze(-1)) * valid
    return torch.stack([nll.sum(), (lse.pow(2) * valid).sum()])


def chunked_loss(h, w, targets, chunk=4096):
    """[1, 3] = [sum of token NLL, number of scored tokens, sum of logsumexp^2].
    Logits are built chunk by chunk (and recomputed in backward), so the full
    [tokens x 32k] logit tensor is never stored."""
    h = h.reshape(-1, h.shape[-1])
    t = targets.reshape(-1)
    parts = []
    for s in range(0, h.shape[0], chunk):
        hs, ts = h[s:s + chunk], t[s:s + chunk]
        if torch.is_grad_enabled() and h.requires_grad:
            parts.append(grad_checkpoint(_ce_chunk, hs, w, ts, use_reentrant=False))
        else:
            parts.append(_ce_chunk(hs, w, ts))
    tot = torch.stack(parts).sum(0)
    count = (t != -100).sum().to(tot.dtype)
    return torch.stack([tot[0], count, tot[1]]).unsqueeze(0)


class EncoderDecoderLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        d, V = config["d_model"], config["vocab_size"]
        self.embed = nn.Embedding(V, d)
        self.enc_blocks = nn.ModuleList([EncoderBlock(config) for _ in range(config["n_enc_layer"])])
        self.enc_norm = RMSNorm(d)
        self.dec_blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config["n_dec_layer"])])
        self.dec_norm = RMSNorm(d)
        self.head = nn.Linear(d, V, bias=False)
        self.head.weight = self.embed.weight                  # tied
        cos, sin = rope_cache(max(config["max_enc_len"], config["max_dec_len"]), config["d_head"], config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.grad_ckpt = False
        self.amp = True                                        # fp16 autocast on GPU (fp32 weights)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
        for name, p in self.named_parameters():                 # scaled residual-output init
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                n_res = 2 * config["n_enc_layer"] if name.startswith("enc_") else 3 * config["n_dec_layer"]
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    def _run(self, blk, *args, **kwargs):
        if self.grad_ckpt and self.training and torch.is_grad_enabled():
            return grad_checkpoint(blk, *args, use_reentrant=False, **kwargs)
        return blk(*args, **kwargs)

    def encode(self, input_ids, attention_mask=None):
        """Contract method: encoder hidden states [B, T, d]. attention_mask
        (1 = real token) marks padding; positions start at 0 in every row
        (pad on the right)."""
        T = input_ids.shape[1]
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = attention_mask.bool() if attention_mask is not None else None
        for blk in self.enc_blocks:
            x = self._run(blk, x, cos, sin, key_mask)
        return self.enc_norm(x)

    def decode_hidden(self, dec_ids, enc_out, enc_mask=None):
        T = dec_ids.shape[1]
        x = self.embed(dec_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        m = enc_mask.bool() if enc_mask is not None else None
        for blk in self.dec_blocks:
            x = self._run(blk, x, cos, sin, enc_out=enc_out, enc_mask=m)
        return self.dec_norm(x)

    def forward(self, enc_ids, dec_ids, targets=None, enc_mask=None):
        """No targets -> logits [B, Q, V]. With targets -> [1, 3] sums
        (NLL, count, lse^2); targets = -100 are ignored."""
        use_amp = enc_ids.is_cuda and self.embed.weight.dtype == torch.float32 and self.amp
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            enc_out = self.encode(enc_ids, enc_mask)
            h = self.decode_hidden(dec_ids, enc_out, enc_mask)
            if targets is None:
                return self.head(h)
            return chunked_loss(h, self.head.weight, targets)

    def decoder_step(self, tok, pos, caches, cross_kvs, enc_mask):
        x = self.embed(tok)
        cos, sin = self.rope_cos[pos:pos + tok.shape[1]], self.rope_sin[pos:pos + tok.shape[1]]
        for blk, cache, kv in zip(self.dec_blocks, caches, cross_kvs):
            x = blk(x, cos, sin, enc_mask=enc_mask, enc_kv=kv, cache=cache)
        return self.head(self.dec_norm(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (one prompt, greedy): returns prompt + continuation ids."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


def build_model(config):
    """Contract function."""
    return EncoderDecoderLM(config)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
def _sample_top_p(logits, temperature, top_p, generator):
    probs = torch.softmax(logits / temperature, dim=-1)
    if top_p < 1.0:
        sorted_p, idx = probs.sort(dim=-1, descending=True)
        drop = (sorted_p.cumsum(-1) - sorted_p) > top_p
        sorted_p = sorted_p.masked_fill(drop, 0.0)
        probs = torch.zeros_like(probs).scatter_(-1, idx, sorted_p)
    return torch.multinomial(probs, 1, generator=generator).squeeze(-1)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_enc_len=768, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None, sample=False,
                   temperature=1.0, top_p=1.0, generator=None, start_id=None):
    """Continuations for several prompts at once. Encoder input is right-padded
    (masked); every decoder row starts at the same position, so no decoder
    padding is needed. Greedy by default; `sample=True` gives temperature /
    top-p sampling (used for MBR candidates). Optional repetition penalty over
    generated tokens, n-gram blocking, and early stop once every row has more
    than `target_words` words."""
    model.eval()
    device = model.embed.weight.device
    cfg = model.config
    start_id = DEC_START_ID if start_id is None else start_id
    max_enc_len = min(max_enc_len or cfg["max_enc_len"], cfg["max_enc_len"])
    prompts = [(p[-max_enc_len:] if p else [EOS_ID if EOS_ID is not None else 0]) for p in prompt_id_lists]
    B, P = len(prompts), max(len(p) for p in prompts)
    enc_ids = torch.zeros(B, P, dtype=torch.long, device=device)
    enc_mask = torch.zeros(B, P, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        enc_ids[b, :len(p)] = torch.tensor(p, device=device)
        enc_mask[b, :len(p)] = True
    use_amp = device.type == "cuda" and model.embed.weight.dtype == torch.float32 and getattr(model, "amp", True)
    amp = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()
    with amp:
        enc_out = model.encode(enc_ids, enc_mask)
        cross_kvs = [blk.cross_attn.project_kv(enc_out) for blk in model.dec_blocks]
        caches = [KVCache(max_new_tokens + 1) for _ in model.dec_blocks]
        tok = torch.full((B, 1), start_id, dtype=torch.long, device=device)
        logits = model.decoder_step(tok, 0, caches, cross_kvs, enc_mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = _sample_top_p(logits, temperature, top_p, generator) if sample else logits.argmax(dim=-1)
        for b, tk in enumerate(nxt.tolist()):
            generated[b].append(tk)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(len(decode(g).split()) > target_words for g in generated):
            break
        with amp:
            logits = model.decoder_step(nxt[:, None], t + 1, caches, cross_kvs, enc_mask)[:, -1, :].float()
    return generated


# ---------------------------------------------------------------------------
# Scoring helpers (mirror evaluate_phase1.py; BERTScore uses roberta-large)
# ---------------------------------------------------------------------------
def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return bin(state).count("1")


def rouge_l_f1(p_text, r_text):
    pt, rt = p_text.split(), r_text.split()
    if not pt or not rt:
        return 0.0
    n = lcs_length(pt, rt)
    if not n:
        return 0.0
    pr, rc = n / len(pt), n / len(rt)
    return 2 * pr * rc / (pr + rc)


def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()


def bleu4(preds, refs):
    clipped, totals, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        pl += len(p)
        rl += len(r)
        for n in range(1, 5):
            pn, rn = _ngrams(p, n), _ngrams(r, n)
            totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x <= 0 for x in prec) or pl == 0:
        return 0.0
    return (1.0 if pl > rl else math.exp(1.0 - rl / pl)) * math.exp(sum(math.log(x) for x in prec) / 4)


def score_predictions(pred, truth, ids):
    try:
        import bert_score
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=True)
        import bert_score
    hyps = [pred[i]["text"].strip() for i in ids]
    refs = [truth[i]["text"].strip() for i in ids]
    _, _, f1 = bert_score.score(hyps, refs, lang="en", batch_size=8, verbose=False,
                                device="cuda" if torch.cuda.is_available() else "cpu")
    return {"bertscore": float(f1.mean()),
            "rouge_l": sum(rouge_l_f1(h, r) for h, r in zip(hyps, refs)) / len(refs),
            "bleu4": bleu4([h.split() for h in hyps], [r.split() for r in refs])}


def to_words(toks, n_words=128):
    words = decode(toks).split()[:n_words]
    return " ".join(words) if words else "."


def predict(model, prompt_dict, ids, batch_size=16, desc="predict", **settings):
    """Greedy/sampled continuations -> {id: {"text": 128 words}}. Prompts are
    sorted by length so each batch has similar lengths."""
    from tqdm.auto import tqdm
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_size), desc=desc, dynamic_ncols=True, mininterval=30):
        chunk = order[s:s + batch_size]
        for i, toks in zip(chunk, generate_batch(model, [enc[i] for i in chunk], **settings)):
            out[i] = {"text": to_words(toks)}
    return {i: out[i] for i in ids}


def mbr_pick(candidates):
    """Minimum-Bayes-risk choice: the candidate with the highest average
    ROUGE-L F1 against the other candidates (a consensus pick; no outside
    model involved)."""
    if len(candidates) <= 2:
        return candidates[0]
    best, best_u = 0, -1.0
    for i, c in enumerate(candidates):
        u = sum(rouge_l_f1(c, o) for j, o in enumerate(candidates) if j != i) / (len(candidates) - 1)
        if u > best_u:
            best, best_u = i, u
    return candidates[best]


def predict_mbr(model, prompt_dict, ids, n_samples=7, batch_prompts=8, greedy_settings=None,
                sample_settings=None, seed=1234, desc="mbr"):
    """For each prompt: 1 greedy candidate + n_samples sampled candidates ->
    MBR pick. Candidates of `batch_prompts` prompts are generated together."""
    from tqdm.auto import tqdm
    device = model.embed.weight.device
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_prompts), desc=desc, dynamic_ncols=True, mininterval=30):
        chunk = order[s:s + batch_prompts]
        greedy = generate_batch(model, [enc[i] for i in chunk], **greedy_settings)
        sampled = generate_batch(model, [enc[i] for i in chunk for _ in range(n_samples)], sample=True,
                                 generator=gen, **sample_settings)
        for k, i in enumerate(chunk):
            cands = [to_words(greedy[k])] + [to_words(t) for t in sampled[k * n_samples:(k + 1) * n_samples]]
            out[i] = {"text": mbr_pick(cands)}
    return {i: out[i] for i in ids}


@torch.no_grad()
def dev_loss_bpb(model, dev_prompts, dev_truth, ids, max_enc_len=768, max_dec_len=512):
    """Continuation loss per token and bits-per-byte (tokenizer-independent)."""
    model.eval()
    device = model.embed.weight.device
    nll, n_tok, n_bytes = 0.0, 0, 0
    for i in ids:
        cont_text = " " + dev_truth[i]["text"].strip()
        cont = encode(cont_text)[:max_dec_len]
        ctx = encode(dev_prompts[i]["text"])[-max_enc_len:]
        if not cont or not ctx:
            continue
        enc_ids = torch.tensor([ctx], device=device)
        dec_in = torch.tensor([[DEC_START_ID] + cont[:-1]], device=device)
        tgt = torch.tensor([cont], device=device)
        out = model(enc_ids, dec_in, targets=tgt)[0]
        nll += out[0].item()
        n_tok += len(cont)
        n_bytes += len(cont_text.encode("utf-8"))
    return nll / max(1, n_tok), nll / (math.log(2) * max(1, n_bytes))


def self_test(device):
    """Checks: cached decoding == full forward; decoder is causal; encoder is
    two-way; encoder padding changes nothing; loss sums are correct; batched
    generation == one-at-a-time; Muon orthogonalises."""
    torch.manual_seed(0)
    tiny = dict(MODEL_D_CONFIG, vocab_size=101, d_model=64, n_enc_layer=2, n_dec_layer=2, n_head=4,
                n_kv_head=2, d_head=16, d_ff=96, max_enc_len=64, max_dec_len=64)
    m = build_model(tiny).to(device).eval()
    m.amp = False                                          # exactness checks in fp32 (GPU kernels still used)
    enc = torch.randint(4, 101, (2, 20), device=device)
    dec = torch.randint(4, 101, (2, 12), device=device)
    with torch.no_grad():
        full = m(enc, dec).float()
        enc_out = m.encode(enc)
        kvs = [blk.cross_attn.project_kv(enc_out) for blk in m.dec_blocks]
        caches = [KVCache(12) for _ in m.dec_blocks]
        steps = [m.decoder_step(dec[:, t:t + 1], t, caches, kvs, None) for t in range(12)]
        diff = (full - torch.cat(steps, 1).float()).abs().max().item()
        dec2 = dec.clone()
        dec2[:, 8] = (dec[:, 8] + 1) % 97 + 4
        leak = (m(enc, dec2)[:, :8] - full[:, :8]).abs().max().item()
        enc2 = enc.clone()
        enc2[:, 15] = (enc[:, 15] + 1) % 97 + 4
        two_way = (m.encode(enc2)[:, 2] - enc_out[:, 2]).abs().max().item()
        padded = torch.cat([enc, torch.zeros(2, 5, dtype=torch.long, device=device)], 1)
        mask = torch.cat([torch.ones(2, 20), torch.zeros(2, 5)], 1).to(device)
        pad_diff = (m(padded, dec, enc_mask=mask) - full).abs().max().item()
        tgt = torch.randint(4, 101, (2, 12), device=device)
        tgt[0, 3] = -100
        s = m(enc, dec, targets=tgt)[0]
        ref = F.cross_entropy(full.reshape(-1, 101), tgt.reshape(-1), ignore_index=-100, reduction="sum")
    assert diff < 1e-3, f"cached decoding != full forward ({diff})"
    assert leak < 1e-5, f"decoder sees future tokens ({leak})"
    assert two_way > 1e-5, "encoder is not bidirectional"
    assert pad_diff < 1e-4, f"encoder padding changes the output ({pad_diff})"
    assert abs(s[0].item() - ref.item()) < 1e-3 and int(s[1].item()) == 23, "loss sums wrong"
    pa, pb = enc[0].tolist(), enc[1, :9].tolist()
    both = generate_batch(m, [pa, pb], max_new_tokens=8, start_id=1)
    alone = [generate_batch(m, [pa], max_new_tokens=8, start_id=1)[0],
             generate_batch(m, [pb], max_new_tokens=8, start_id=1)[0]]
    assert both == alone, "batched generation differs from one-at-a-time"
    g = torch.Generator(device=device)
    g.manual_seed(0)
    smp = generate_batch(m, [pa, pb], max_new_tokens=8, start_id=1, sample=True, temperature=0.8, top_p=0.9,
                         generator=g)
    assert len(smp) == 2 and all(len(x) == 8 for x in smp)
    X = newton_schulz5(torch.randn(48, 96, device=device)).float()
    sv = torch.linalg.svdvals(X)
    assert 0.3 < sv.min().item() and sv.max().item() < 1.5, f"Muon orthogonalisation off: {sv.min():.2f}..{sv.max():.2f}"
    print(f"Self-test passed: cached==full {diff:.1e}, causal decoder {leak:.1e}, two-way encoder, "
          f"padding-safe {pad_diff:.1e}, loss sums ok, batched==single, sampling ok, "
          f"Muon singular values {sv.min():.2f}..{sv.max():.2f}.")


# ---------------------------------------------------------------------------
# Muon optimizer (Jordan et al. 2024; update scale from Liu et al. 2025,
# "Muon is Scalable for LLM Training": 0.2 * sqrt(max(rows, cols)) so the
# learning rate and weight decay work on AdamW's scale).
# ---------------------------------------------------------------------------
def newton_schulz5(G, steps=5, eps=1e-7):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    X = X / (X.norm() + eps)                              # normalise in fp32 (tiny grads survive)
    if G.is_cuda:
        X = X.half()                                       # T4: fast fp16 matmuls, values stay <= 1
    transposed = X.shape[0] > X.shape[1]
    if transposed:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        Bm = b * A + c * (A @ A)
        X = a * X + Bm @ X
    if transposed:
        X = X.T
    return X.to(G.dtype)


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, momentum=0.95, nesterov=True, weight_decay=0.1, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov,
                                      weight_decay=weight_decay, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, mom, wd = group["lr"], group["momentum"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad.float()
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(g)
                buf = state["momentum_buffer"]
                buf.mul_(mom).add_(g)
                g = g.add(buf, alpha=mom) if group["nesterov"] else buf
                u = newton_schulz5(g, steps=group["ns_steps"]).float()
                p.mul_(1.0 - lr * wd)
                p.add_(u, alpha=-lr * 0.2 * math.sqrt(max(p.shape[0], p.shape[1])))
        return None


def build_optimizers(model, kind, lr, weight_decay):
    muon_params, adam_decay, adam_no_decay = [], [], []
    for name, p in model.named_parameters():
        if p.dim() == 2 and (name.startswith("enc_blocks") or name.startswith("dec_blocks")) and kind == "muon":
            muon_params.append(p)
        elif p.dim() >= 2:
            adam_decay.append(p)                           # embedding (tied output head); all matrices for AdamW
        else:
            adam_no_decay.append(p)                        # norm weights
    adam = torch.optim.AdamW([{"params": adam_decay, "weight_decay": weight_decay},
                              {"params": adam_no_decay, "weight_decay": 0.0}], lr=lr, betas=(0.9, 0.95), eps=1e-8)
    return ([Muon(muon_params, lr=lr, weight_decay=weight_decay)] if muon_params else []) + [adam]


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def lr_factor(step, progress, warmup_steps, cooldown_frac, floor):
    """Warmup-stable-decay on *time*: linear warmup (steps), constant, then a
    1-sqrt cooldown over the last `cooldown_frac` of the training time."""
    w = min(1.0, (step + 1) / max(1, warmup_steps))
    start = 1.0 - cooldown_frac
    if progress < start:
        return w
    r = min(1.0, (progress - start) / max(1e-9, cooldown_frac))
    return w * (floor + (1.0 - floor) * (1.0 - math.sqrt(r)))


def loss_from(out, z_loss_coef):
    s = out.sum(0)
    n = s[1].clamp(min=1)
    return s[0] / n, (s[0] + z_loss_coef * s[2]) / n


# ---------------------------------------------------------------------------
# Data: stream of 1024-token windows (round-robin over the cleaned files,
# documents separated by <eos>), byte positions saved for resume. Each GPU
# (rank) starts at a different offset of every file.
# ---------------------------------------------------------------------------
class WindowStream:
    def __init__(self, file_paths, window, rank=0, world=1, start_positions=None):
        self.files = [Path(p) for p in file_paths if Path(p).stat().st_size > 0]
        self.window = window
        self.positions, self.unaligned = {}, set()
        for p in self.files:
            if start_positions and p.name in start_positions:
                self.positions[p.name] = int(start_positions[p.name])
            else:
                self.positions[p.name] = (p.stat().st_size * rank) // world
                if self.positions[p.name] > 0:
                    self.unaligned.add(p.name)

    def __iter__(self):
        buffer = []
        readers = [self._read_lines(p) for p in self.files]
        for reader in itertools.cycle(readers):
            ids = encode(next(reader))
            if not ids:
                continue
            buffer.extend(ids)
            buffer.append(EOS_ID)
            while len(buffer) >= self.window:
                yield buffer[:self.window]
                buffer = buffer[self.window:]

    def _read_lines(self, path):
        name, size = path.name, path.stat().st_size
        while True:
            start = self.positions[name] if self.positions[name] < size else 0
            with open(path, "rb") as f:
                f.seek(start)
                pos = start
                if name in self.unaligned:                  # started mid-line: skip to the next line
                    pos += len(f.readline())
                    self.unaligned.discard(name)
                while True:
                    raw = f.readline()
                    if not raw:
                        break
                    pos += len(raw)
                    self.positions[name] = pos
                    line = raw.decode("utf-8", errors="ignore").strip()
                    if line:
                        yield line
            self.positions[name] = 0


def make_example_batch(windows, P, Q, device):
    """windows: list of 1024-token lists -> encoder ids [B,P], decoder input
    [B,Q] (start token + shifted continuation), targets [B,Q]."""
    w = torch.tensor(windows, dtype=torch.long)
    enc = w[:, :P]
    tgt = w[:, P:P + Q]
    dec_in = torch.cat([torch.full((w.shape[0], 1), DEC_START_ID, dtype=torch.long), tgt[:, :-1]], dim=1)
    return enc.to(device, non_blocking=True), dec_in.to(device, non_blocking=True), tgt.to(device, non_blocking=True)


def choose_shape(shapes, seed, step, micro):
    """Same (prompt, continuation) split on every GPU for a given micro-step."""
    rnd = random.Random(seed * 1_000_003 + step * 97 + micro)
    return rnd.choices([(s[0], s[1]) for s in shapes], weights=[s[2] for s in shapes])[0]


# ---------------------------------------------------------------------------
# Checkpoints and Kaggle push
# ---------------------------------------------------------------------------
def step_of(path):
    return int(Path(path).stem.split("_step")[1].split("_tokens")[0])


def find_latest_checkpoint(prefix, search_dirs, extract_to):
    cands = []
    for d in search_dirs:
        d = Path(d)
        if d.exists():
            cands += [p for p in d.rglob(f"{prefix}_step*.pt")]
    best_zip = None
    for d in search_dirs:
        d = Path(d)
        if not d.exists():
            continue
        for zpath in d.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zpath) as zf:
                    for name in zf.namelist():
                        base = Path(name).name
                        if base.startswith(f"{prefix}_step") and base.endswith(".pt"):
                            if best_zip is None or step_of(base) > best_zip[0]:
                                best_zip = (step_of(base), zpath, name)
            except zipfile.BadZipFile:
                continue
    best_file = max(cands, key=step_of) if cands else None
    if best_zip is not None and (best_file is None or best_zip[0] > step_of(best_file)):
        _, zpath, name = best_zip
        dest = Path(extract_to) / Path(name).name
        with zipfile.ZipFile(zpath) as zf, zf.open(name) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)
        return dest
    return best_file


def prune_old_checkpoints(ckpt_dir, prefix, keep=2):
    ckpts = sorted(Path(ckpt_dir).glob(f"{prefix}_step*.pt"), key=step_of)
    for p in ckpts[:-keep] if len(ckpts) > keep else []:
        p.unlink()


def push_checkpoint_dataset(output_dir, dataset_id, message):
    kdir = Path.home() / ".kaggle"
    if not ((kdir / "access_token").exists() or (kdir / "kaggle.json").exists() or os.environ.get("KAGGLE_API_TOKEN")):
        print("Skipping checkpoint push: no Kaggle API token.", flush=True)
        return False
    slug = dataset_id.split("/")[1]
    with open(Path(output_dir) / "dataset-metadata.json", "w") as f:
        json.dump({"title": slug, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}, f)
    try:
        exists = subprocess.run(["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True,
                                timeout=300).returncode == 0
        cmd = (["kaggle", "datasets", "version", "-p", str(output_dir), "-m", message, "-r", "zip"] if exists
               else ["kaggle", "datasets", "create", "-p", str(output_dir), "-r", "zip"])
        t0 = time.time()
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    except (OSError, subprocess.TimeoutExpired) as exc:
        print(f"Checkpoint push failed: {exc}", flush=True)
        return False
    if result.returncode != 0:
        print(f"Checkpoint push failed: {(result.stderr or result.stdout).strip()[-500:]}", flush=True)
        return False
    print(f"Pushed checkpoint dataset in {time.time() - t0:.0f}s ({message}).", flush=True)
    return True


def load_for_inference(path, device, half=False):
    """fp32 weights + fp16 autocast on GPU (same numerics as training)."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = build_model(ckpt["config"])
    model.load_state_dict(ckpt["model_state"])
    meta = {k: ckpt.get(k) for k in ("step", "tokens_seen", "config", "run_id")}
    del ckpt
    model = model.to(device).eval()
    if half and device.type == "cuda":
        model = model.half()
        model.rope_cos, model.rope_sin = model.rope_cos.float(), model.rope_sin.float()
    return model, meta


# ---------------------------------------------------------------------------
# Trainer (one process per GPU)
# ---------------------------------------------------------------------------
def _probe_micro_batch(model, cfg, device, P, Q):
    """Largest micro-batch (1024-token examples) that fits the memory budget
    on this GPU, with the real optimizers; weights are restored afterwards."""
    saved = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}
    results = {}
    for bs in cfg["candidate_micro_bs"]:
        optims = build_optimizers(model, cfg["optimizer"], 1e-5, 0.0)
        scaler = make_grad_scaler(True)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        try:
            windows = [[random.randrange(4, 1000) for _ in range(P + Q)] for _ in range(bs)]
            enc, dec_in, tgt = make_example_batch(windows, P, Q, device)
            times = []
            for _ in range(3):
                torch.cuda.synchronize(device)
                t0 = time.time()
                for o in optims:
                    o.zero_grad(set_to_none=True)
                _, total = loss_from(model(enc, dec_in, targets=tgt), 1e-4)
                scaler.scale(total).backward()
                for o in optims:
                    scaler.step(o)
                scaler.update()
                torch.cuda.synchronize(device)
                times.append(time.time() - t0)
            peak = torch.cuda.max_memory_allocated(device) / 1e9
            ok = peak <= cfg["memory_budget_gb"]
            results[bs] = (peak, min(times[1:]), ok)
            if not ok:
                break
        except Exception as exc:                           # noqa: BLE001
            if "out of memory" not in str(exc).lower():
                raise
            results[bs] = (None, None, False)
            break
        finally:
            for o in optims:
                o.zero_grad(set_to_none=True)
            del optims
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    model.load_state_dict(saved)
    return results


def train_main(config_path):
    cfg = json.load(open(config_path))
    distributed = int(os.environ.get("WORLD_SIZE", "1")) > 1
    cuda = torch.cuda.is_available()
    if distributed:
        dist.init_process_group("nccl" if cuda else "gloo", timeout=datetime.timedelta(minutes=90))
        rank, world, local_rank = dist.get_rank(), dist.get_world_size(), int(os.environ.get("LOCAL_RANK", 0))
    else:
        rank, world, local_rank = 0, 1, 0
    device = torch.device(f"cuda:{local_rank}") if cuda else torch.device("cpu")
    if cuda:
        torch.cuda.set_device(device)
        torch.backends.cuda.matmul.allow_tf32 = True
    main = rank == 0

    def log(*a):
        if main:
            print(*a, flush=True)

    load_tokenizer(cfg["tokenizer_path"])
    model_cfg = cfg["model"]
    prefix = cfg["ckpt_prefix"]
    ckpt_dir = Path(cfg["checkpoint_dir"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    session_start, session_deadline = cfg["session_start_epoch"], cfg["session_start_epoch"] + 60 * cfg["time_budget_min"]
    total_seconds = 3600 * cfg["total_train_hours"]
    log(f"[D] world={world} device={device} | budget this session {cfg['time_budget_min']} min, "
        f"whole run {cfg['total_train_hours']} h | optimizer={cfg['optimizer']}")

    torch.manual_seed(cfg["seed"])
    random.seed(cfg["seed"] + rank)
    model = build_model(model_cfg).to(device)
    n_params = count_params(model)
    log(f"[D] parameters: {n_params / 1e6:.1f}M")

    # ---- resume ----------------------------------------------------------
    step, tokens_seen, seconds_before, rank_positions, ckpt = 0, 0, 0.0, None, None
    phase = cfg.get("phase_id", "main")        # a new phase id = continue training with a fresh LR schedule
    phase_start_step = 0
    latest = find_latest_checkpoint(prefix, [ckpt_dir] + cfg.get("resume_search_dirs", []), ckpt_dir) if main else None
    if distributed:
        box = [str(latest) if latest else ""]
        dist.broadcast_object_list(box, src=0)
        latest = Path(box[0]) if box[0] else None
    if latest is not None:
        ckpt = torch.load(latest, map_location="cpu", weights_only=False)
        assert ckpt["config"] == model_cfg, "Checkpoint config does not match MODEL_D_CONFIG"
        model.load_state_dict(ckpt["model_state"])
        step, tokens_seen = ckpt["step"], ckpt["tokens_seen"]
        same_session = ckpt.get("session_start_epoch") == session_start
        rank_positions = ckpt.get("data_state", {}).get("rank_positions")
        ckpt_phase = ckpt.get("phase_id", "main")
        if ckpt_phase == phase:
            seconds_before = ckpt["seconds_before_session"] if same_session else ckpt["progress_seconds"]
            phase_start_step = ckpt.get("phase_start_step", 0)
        else:
            seconds_before, phase_start_step = 0.0, step
            log(f"[D] new training phase '{phase}' (previous: '{ckpt_phase}') -> fresh LR schedule from step {step:,}")
        log(f"[D] resumed from {latest.name}: step={step:,} tokens={tokens_seen:,} "
            f"phase progress={seconds_before / total_seconds:.1%} ({'same' if same_session else 'new'} session)")
    else:
        log("[D] no checkpoint found -- starting fresh")

    # ---- micro-batch size ---------------------------------------------------
    shapes = cfg["shapes"]
    Pp, Qp = max(((s[0], s[1]) for s in shapes), key=lambda s: s[1])      # longest decoder = most memory
    if cfg.get("fixed_micro_bs") or not cuda:
        micro_bs, sec_micro = cfg.get("fixed_micro_bs") or 2, None
    else:
        res = _probe_micro_batch(model, cfg, device, Pp, Qp)
        fits = [bs for bs, (_, _, ok) in res.items() if ok]
        if not fits:
            model.grad_ckpt = True
            log("[D] nothing fit without activation checkpointing -> turning it on")
            res = _probe_micro_batch(model, cfg, device, Pp, Qp)
            fits = [bs for bs, (_, _, ok) in res.items() if ok]
        for bs, (peak, sec, ok) in res.items():
            log(f"  micro-batch {bs:3d} x 1024 -> " + (f"{peak:5.2f} GB, {sec:.2f}s, {bs * 1024 / sec:,.0f} tok/s/GPU"
                                                      if peak else "OOM") + ("" if ok else "  [no]"))
        best = max(fits) if fits else 0
        if distributed:
            t = torch.tensor([best], device=device)
            dist.all_reduce(t, op=dist.ReduceOp.MIN)
            best = int(t.item())
        assert best > 0, "No micro-batch fits in memory"
        micro_bs, sec_micro = best, res[best][1]
    accum = max(1, round(cfg["tokens_per_step_target"] / (micro_bs * 1024 * world)))
    tokens_per_step = micro_bs * 1024 * world * accum
    log(f"[D] micro-batch {micro_bs} x 1024 tokens x {world} GPU(s) x accum {accum} = {tokens_per_step:,} tokens/step"
        + (f" | probe ~{micro_bs * 1024 * world / sec_micro:,.0f} tok/s" if sec_micro else ""))

    optims = build_optimizers(model, cfg["optimizer"], cfg["peak_lr"], cfg["weight_decay"])
    scaler = make_grad_scaler(cuda)
    if ckpt is not None:
        if ckpt.get("optimizer_kind") == cfg["optimizer"]:
            for o, s in zip(optims, ckpt["optim_states"]):
                o.load_state_dict(s)
            scaler.load_state_dict(ckpt["scaler_state"])
        else:
            log("[D] optimizer changed since the checkpoint -> fresh optimizer state")
        model.grad_ckpt = model.grad_ckpt or ckpt.get("grad_ckpt", False)
        del ckpt
    train_model = model
    if distributed:
        from torch.nn.parallel import DistributedDataParallel as DDP
        train_model = DDP(model, device_ids=[local_rank] if cuda else None, gradient_as_bucket_view=True)

    positions = rank_positions[rank] if rank_positions and rank < len(rank_positions) else None
    files = sorted(Path(cfg["data_dir"]).glob("clean_*.txt"))
    assert files, "No clean_*.txt files in the data folder"
    stream = WindowStream(files, 1024, rank, world, positions)
    data_iter = iter(stream)
    log(f"[D] training files: {[p.name for p in files]} | shapes (prompt, continuation, weight): {shapes}")

    def save_checkpoint(reason):
        gathered = [None] * world
        if distributed:
            dist.all_gather_object(gathered, dict(stream.positions))
        else:
            gathered = [dict(stream.positions)]
        path = None
        if main:
            now = time.time()
            state = {
                "model_state": model.state_dict(), "optim_states": [o.state_dict() for o in optims],
                "optimizer_kind": cfg["optimizer"], "scaler_state": scaler.state_dict(), "step": step,
                "tokens_seen": tokens_seen, "config": model_cfg, "arch_id": ARCH_ID, "run_id": cfg["run_id"],
                "session_start_epoch": session_start, "seconds_before_session": seconds_before,
                "progress_seconds": seconds_before + (now - session_start), "grad_ckpt": model.grad_ckpt,
                "phase_id": phase, "phase_start_step": phase_start_step,
                "data_state": {"rank_positions": gathered, "world": world},
                "train_cfg": {k: cfg[k] for k in ("peak_lr", "shapes", "optimizer", "total_train_hours")},
                "reason": reason,
            }
            path = ckpt_dir / f"{prefix}_step{step}_tokens{tokens_seen}.pt"
            tmp = path.with_name(path.name + ".tmp")
            torch.save(state, tmp)
            tmp.replace(path)
            prune_old_checkpoints(ckpt_dir, prefix, keep=2)
        if distributed:
            dist.barrier()
        return path

    # ---- loop --------------------------------------------------------------
    train_model.train()
    t_loop, last_ckpt, last_log_t, last_log_tokens = time.time(), time.time(), time.time(), tokens_seen
    start_tokens = tokens_seen
    bad_streak, skipped, stop_reason = 0, 0, "unknown"
    ce_log, n_log = torch.zeros((), device=device), 0
    flags = torch.zeros(3, device=device)
    while True:
        if main:
            now = time.time()
            progress = (seconds_before + (now - session_start)) / total_seconds
            stop = 1.0 if (progress >= 1.0 or now >= session_deadline) else 0.0
            do_ckpt = 1.0 if (now - last_ckpt) / 60 >= cfg["checkpoint_every_min"] else 0.0
            flags = torch.tensor([progress, stop, do_ckpt], device=device)
        if distributed:
            dist.broadcast(flags, src=0)
        progress, stop, do_ckpt = float(flags[0]), bool(flags[1] > 0.5), bool(flags[2] > 0.5)
        if stop:
            stop_reason = "schedule complete" if progress >= 1.0 else "session time budget reached"
            break
        lr = cfg["peak_lr"] * lr_factor(step - phase_start_step, progress, cfg["warmup_steps"], cfg["cooldown_frac"], cfg["min_lr_frac"])
        for o in optims:
            for g in o.param_groups:
                g["lr"] = lr
            o.zero_grad(set_to_none=True)
        for micro in range(accum):
            P, Q = choose_shape(shapes, cfg["seed"], step, micro)
            enc, dec_in, tgt = make_example_batch([next(data_iter) for _ in range(micro_bs)], P, Q, device)
            sync_ctx = train_model.no_sync() if (distributed and micro < accum - 1) else contextlib.nullcontext()
            with sync_ctx:
                ce, total = loss_from(train_model(enc, dec_in, targets=tgt), cfg["z_loss_coef"])
                scaler.scale(total / accum).backward()
            ce_log += ce.detach() / accum
            tokens_seen += micro_bs * 1024 * world
        for o in optims:
            scaler.unscale_(o)
        gnorm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"]))
        for o in optims:
            scaler.step(o)
        scaler.update()
        n_log += 1
        step += 1
        if math.isfinite(gnorm):
            bad_streak = 0
        else:
            bad_streak += 1
            skipped += 1
            if bad_streak >= 50:
                stop_reason = "50 non-finite steps in a row"
                break
        if distributed and step % cfg["sync_every"] == 0:
            for p in model.parameters():                     # remove any tiny drift between GPUs
                dist.broadcast(p.data, src=0)
        if step % cfg["log_every"] == 0 or step == 1:
            if distributed:
                dist.all_reduce(ce_log)
                ce_log /= world
            now = time.time()
            tps = (tokens_seen - last_log_tokens) / max(1e-6, now - last_log_t)
            log(f"  step {step:6d}  loss={ce_log.item() / max(1, n_log):.4f}  lr={lr:.2e}  gnorm={gnorm:.2f}  "
                f"tokens={tokens_seen:,}  tok/s={tps:,.0f}  elapsed={(now - t_loop) / 60:.1f} min  "
                f"progress={progress:.1%}  [{datetime.datetime.now():%H:%M:%S}]")
            ce_log.zero_()
            n_log, last_log_t, last_log_tokens = 0, now, tokens_seen
        if do_ckpt:
            path = save_checkpoint("periodic")
            last_ckpt = time.time()
            if main:
                print(f"  [checkpoint saved -> {path.name}]", flush=True)
                if cfg["push"]:
                    push_checkpoint_dataset(cfg["output_dir"], cfg["dataset_id"],
                                            f"{cfg['run_id']}: step={step}, tokens={tokens_seen:,} (in progress)")
            if distributed:
                dist.barrier()
    path = save_checkpoint(stop_reason)
    loop_min = (time.time() - t_loop) / 60
    summary = {"stop_reason": stop_reason, "step": step, "tokens_seen": tokens_seen,
               "session_tokens": tokens_seen - start_tokens, "loop_minutes": round(loop_min, 1),
               "tokens_per_sec": round((tokens_seen - start_tokens) / max(1e-6, loop_min * 60), 1),
               "world": world, "micro_bs": micro_bs, "accum": accum, "tokens_per_step": tokens_per_step,
               "skipped_steps": skipped, "grad_ckpt": model.grad_ckpt, "n_params": n_params, "phase_id": phase,
               "phase_start_step": phase_start_step,
               "final_checkpoint": path.name if path else None, "progress": round(progress, 4),
               "peak_mem_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if cuda else 0.0}
    if main:
        json.dump(summary, open(Path(cfg["output_dir"]) / "train_summary.json", "w"), indent=1)
        log(f"[D] stopped ({stop_reason}) at step={step:,}, tokens={tokens_seen:,} after {loop_min:.1f} min "
            f"-> {path.name}")
    if distributed:
        dist.barrier()
        dist.destroy_process_group()
    if stop_reason.startswith("50 non-finite"):
        sys.exit(3)


if __name__ == "__main__":
    train_main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/c_model.py
"""c_model.py -- Model C (prefix-LM) classes, copied from p1_07_model_C.ipynb."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

decode = None          # only used by C's text generation, not needed here


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(seq_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x [B,H,T,d]; cos/sin [B or 1, T, d/2]. Rotation in fp32."""
    x1, x2 = x[..., 0::2].float(), x[..., 1::2].float()
    cos, sin = cos[:, None].float(), sin[:, None].float()
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).to(x.dtype)


def prefix_lm_mask(prefix_len, T, device):
    """Bool [B, 1, T, T]: key k visible to query q iff k <= q or k < prefix_len[b]."""
    idx = torch.arange(T, device=device)
    causal = idx[None, :, None] >= idx[None, None, :]
    in_prefix = idx[None, None, :] < prefix_len[:, None, None]
    return (causal | in_prefix)[:, None]


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh, b = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"], cfg["bias"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=b)
        self.k_proj = nn.Linear(d, kv * dh, bias=b)
        self.v_proj = nn.Linear(d, kv * dh, bias=b)
        self.o_proj = nn.Linear(h * dh, d, bias=b)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class GeluMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.up_proj = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["bias"])
        self.down_proj = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = GeluMLP(cfg)

    def _inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        return x + self.mlp(self.mlp_norm(x))

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._inner(x, cos, sin, attn_mask, cache)


class KVCache:
    def __init__(self, batch_size, n_kv_head, max_len, d_head, device, dtype):
        self.k = torch.zeros(batch_size, n_kv_head, max_len, d_head, device=device, dtype=dtype)
        self.v = torch.zeros_like(self.k)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class PrefixLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([Block(config) for _ in range(config["n_layer"])])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight
        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def encode(self, input_ids, attention_mask=None, mode="prefix", prefix_len=None):
        """Contract method: hidden states. mode 'causal' (prefix 0), 'prefix'
        (needs prefix_len [B]) or 'bidirectional' (whole input is prefix)."""
        B, T = input_ids.shape
        if mode == "causal" or prefix_len is None and mode == "prefix":
            prefix_len = torch.zeros(B, dtype=torch.long, device=input_ids.device)
        elif mode == "bidirectional":
            prefix_len = torch.full((B,), T, dtype=torch.long, device=input_ids.device)
        mask = prefix_lm_mask(prefix_len, T, input_ids.device) if bool((prefix_len > 0).any()) else None
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[None, :T], self.rope_sin[None, :T]
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, mask, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, prefix_len=None, targets=None):
        """No targets -> logits. With targets -> [1, 3] = [loss sum, number of
        loss tokens, sum of logsumexp^2 (z-loss)] over positions that predict
        continuation tokens (t >= prefix_len - 1). Sums (not means) so
        DataParallel replicas combine exactly."""
        use_amp = self.training and ids.is_cuda
        if prefix_len is None:
            prefix_len = torch.zeros(ids.shape[0], dtype=torch.long, device=ids.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = self.head(self.encode(ids, mode="prefix", prefix_len=prefix_len))
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        nll = lse - logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        pos = torch.arange(ids.shape[1], device=ids.device)[None, :]
        w = (pos >= (prefix_len[:, None] - 1)).float()
        return torch.stack([(nll * w).sum(), w.sum(), (lse.pow(2) * w).sum()]).unsqueeze(0)

    def step(self, ids, positions, caches, attn_mask):
        """Generation step with a KV cache. ids/positions [B, T]; attn_mask
        bool [B, 1, T, cache_len + T]."""
        x = self.embed(ids)
        cos, sin = self.rope_cos[positions], self.rope_sin[positions]
        for blk, cache in zip(self.blocks, caches):
            x = blk(x, cos, sin, attn_mask, cache=cache)
        return self.head(self.norm_f(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (greedy, one prompt): the prompt is the prefix."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_context=None, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None):
    """Greedy continuations for several prompts at once (left-padded). The
    prompt is the bidirectional prefix; continuation is causal. Optional:
    repetition penalty over generated tokens, n-gram blocking inside the
    continuation, stop once every row has > target_words words."""
    model.eval()
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    cfg = model.config
    max_context = max_context or cfg["max_seq_len"]
    prompts = [p[-(max_context - max_new_tokens):] for p in prompt_id_lists]
    B, T0 = len(prompts), max(len(p) for p in prompts)
    pad = torch.tensor([T0 - len(p) for p in prompts], device=device)
    ids = torch.zeros(B, T0, dtype=torch.long, device=device)
    for b, p in enumerate(prompts):
        ids[b, T0 - len(p):] = torch.tensor(p, device=device)
    total = T0 + max_new_tokens
    caches = [KVCache(B, cfg["n_kv_head"], total, cfg["d_head"], device, dtype) for _ in range(cfg["n_layer"])]
    idx = torch.arange(total, device=device)
    key_ok = idx[None, :] >= pad[:, None]
    positions = (idx[None, :] - pad[:, None]).clamp(min=0)
    # Prefill: the prompt attends to itself in both directions (no padding keys);
    # every query may see itself so padded rows are never fully masked.
    mask = (key_ok[:, None, :T0] | torch.eye(T0, dtype=torch.bool, device=device)[None])[:, None]
    logits = model.step(ids, positions[:, :T0], caches, mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")     # ids the tokenizer cannot decode
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = logits.argmax(dim=-1)
        for b, tok in enumerate(nxt.tolist()):
            generated[b].append(tok)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(
                len(decode(g).split()) > target_words for g in generated):
            break
        cur = T0 + t
        logits = model.step(nxt[:, None], positions[:, cur:cur + 1], caches,
                            key_ok[:, None, None, :cur + 1])[:, -1, :].float()
    return generated


def build_model(config):
    """Contract function."""
    return PrefixLM(config)


USE_GRAD_CHECKPOINTING = True

In [ ]:
%%writefile /kaggle/working/a_model.py
"""a_model.py -- Model A (Llama-style decoder) classes, copied from p1_03_model_A.ipynb."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
class RMSNorm(nn.Module):
    """RMSNorm instead of LayerNorm: one learned scale, no learned bias,
    no mean-subtraction -- cheaper, and the modern-architecture standard
    this whole design follows (Llama, Mistral, etc.).
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


try:
    _probe = torch.randn(1, 1, 2, 4)
    F.scaled_dot_product_attention(_probe, _probe, _probe, enable_gqa=True)
    _SDPA_HAS_ENABLE_GQA = True
except TypeError:
    _SDPA_HAS_ENABLE_GQA = False


class GQAAttention(nn.Module):
    """Grouped-query attention: fewer KV heads than query heads, which
    shrinks the KV cache proportionally (here, 7x) at a small quality cost.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)

    def forward(self, x, cos, sin, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if cache is not None:
            k, v = cache.update(k, v)

        is_causal = q.shape[2] > 1

        if _SDPA_HAS_ENABLE_GQA:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal, enable_gqa=True)
        else:
            n_rep = self.n_head // self.n_kv_head
            k_rep = k.repeat_interleave(n_rep, dim=1)
            v_rep = v.repeat_interleave(n_rep, dim=1)
            out = F.scaled_dot_product_attention(q, k_rep, v_rep, is_causal=is_causal)

        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    """One transformer layer: pre-norm attention, pre-norm MLP, both as
    residual additions.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head, d_ff):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head, d_head)
        self.mlp_norm = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, d_ff)

    def _forward_inner(self, x, cos, sin, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x

    def forward(self, x, cos, sin, cache=None, use_checkpoint=False):
        # Checkpointing needs a cache-free forward (no in-place cache
        # mutation across a recomputed backward pass) and only makes
        # sense during training, never during KV-cache generation.
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._forward_inner, x, cos, sin, None, use_reentrant=False)
        return self._forward_inner(x, cos, sin, cache)


class KVCache:
    """A pre-allocated, fixed-size cache for one layer's keys and values."""
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config["d_model"], config["n_head"], config["n_kv_head"],
                  config["d_head"], config["d_ff"])
            for _ in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # weight tying

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def new_kv_caches(self, batch_size, device, dtype):
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def encode(self, input_ids, attention_mask=None, mode="causal"):
        """Contract method: raw hidden states, no LM head. attention_mask
        is accepted for interface compatibility with later phases (e.g. a
        padded batch) but unused here -- Phase 1 pretraining always packs
        full, unpadded windows, so every position is valid.
        """
        assert mode == "causal", "MiniGPT is decoder-only; only causal encoding is supported."
        B, T = input_ids.shape
        x = self.embed(input_ids)
        cos = self.rope_cos[:T].to(x.device)
        sin = self.rope_sin[:T].to(x.device)
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, cache=None, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, caches=None):
        if caches is None:
            return self.head(self.encode(ids))
        # Generation path (KV cache present): identical to encode() but
        # threading a per-layer cache through and offsetting RoPE by how
        # much of the cache is already filled.
        B, T = ids.shape
        x = self.embed(ids)
        pos_offset = caches[0].pos
        cos = self.rope_cos[pos_offset:pos_offset + T].to(x.device)
        sin = self.rope_sin[pos_offset:pos_offset + T].to(x.device)
        for i, blk in enumerate(self.blocks):
            x = blk(x, cos, sin, cache=caches[i], use_checkpoint=False)
        x = self.norm_f(x)
        return self.head(x)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", top_p=0.9, temperature=1.0):
        """Contract method. `sampler`: "greedy" (argmax, deterministic --
        what the smoke-test submission uses) or "top_p" (nucleus sampling).
        Stops early (per sequence, via a done mask) at <eos> or max_seq_len.
        """
        assert kv_cache, "This notebook's generate() only implements the KV-cache path -- it's what training will actually ship."
        self.eval()
        device, dtype = input_ids.device, next(self.parameters()).dtype

        # A prompt longer than the model's context has nowhere to put
        # max_new_tokens (or, if it's longer than max_seq_len outright,
        # RoPE's precomputed cos/sin tables don't even cover it -- a hard
        # crash, not a quality issue). Truncate to the most recent tokens
        # -- the ones closest to where generation continues -- rather
        # than letting this crash the whole generation loop over one long
        # dev-set example.
        max_prompt_len = self.config["max_seq_len"] - max_new_tokens
        if input_ids.shape[1] > max_prompt_len:
            print(f"  [generate() warning: prompt has {input_ids.shape[1]} tokens, "
                  f"truncating to the last {max_prompt_len} to leave room for "
                  f"{max_new_tokens} new tokens within max_seq_len={self.config['max_seq_len']}]")
            input_ids = input_ids[:, -max_prompt_len:]

        B, T0 = input_ids.shape
        max_total = min(T0 + max_new_tokens, self.config["max_seq_len"])

        caches = self.new_kv_caches(batch_size=B, device=device, dtype=dtype)
        generated = input_ids
        done = torch.zeros(B, dtype=torch.bool, device=device)

        logits = self.forward(input_ids, caches=caches)  # prefill: process the whole prompt at once
        next_ids = self._sample(logits[:, -1, :], sampler, top_p, temperature)

        while generated.shape[1] < max_total:
            next_ids = torch.where(done, torch.full_like(next_ids, EOS_ID), next_ids)
            generated = torch.cat([generated, next_ids.unsqueeze(1)], dim=1)
            done = done | (next_ids == EOS_ID)
            if done.all():
                break
            logits = self.forward(next_ids.unsqueeze(1), caches=caches)  # one new token at a time
            next_ids = self._sample(logits[:, -1, :], sampler, top_p, temperature)

        self.train()
        return generated

    @staticmethod
    def _sample(logits, sampler, top_p, temperature):
        if sampler == "greedy":
            return logits.argmax(dim=-1)
        probs = F.softmax(logits / temperature, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cumulative = sorted_probs.cumsum(dim=-1)
        cutoff = (cumulative <= top_p).sum(dim=-1, keepdim=True).clamp(min=1)
        mask = torch.arange(sorted_probs.shape[-1], device=logits.device)[None, :] < cutoff
        sorted_probs = sorted_probs * mask
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        picked = torch.multinomial(sorted_probs, num_samples=1).squeeze(-1)
        return sorted_idx.gather(-1, picked.unsqueeze(-1)).squeeze(-1)


def build_model(config):
    """Contract function: the one place that turns a config dict into a
    model instance, so every later phase constructs Model A the same way.
    """
    return MiniGPT(config)


USE_GRAD_CHECKPOINTING = True
EOS_ID = None

In [ ]:
%%writefile /kaggle/working/b_model.py
"""b_model.py -- Model B (local/global decoder) classes, copied from p1_04_model_B.ipynb."""
import math
Z_LOSS_COEF = 1e-4
EOS_ID = None
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :].to(x.dtype)
    sin = sin[None, None, :, :].to(x.dtype)
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


_TRAIN_MASK_CACHE = {}


def sliding_window_mask(q_start, n_q, n_k, window, device, cacheable=False):
    """Boolean [n_q, n_k] mask, True = may attend. Query i sits at absolute
    position q_start + i; key j at position j. Allowed iff the key is not in
    the future AND is less than `window` positions back.
    Only training masks are cached (training always uses one fixed shape);
    eval/generation shapes vary per prompt, so caching them would leak memory.
    """
    cache_key = (n_q, n_k, window, str(device))
    if cacheable and cache_key in _TRAIN_MASK_CACHE:
        return _TRAIN_MASK_CACHE[cache_key]
    q_pos = torch.arange(q_start, q_start + n_q, device=device)[:, None]
    k_pos = torch.arange(n_k, device=device)[None, :]
    allowed = (k_pos <= q_pos) & (q_pos - k_pos < window)
    if cacheable:
        _TRAIN_MASK_CACHE[cache_key] = allowed
    return allowed


class LocalGlobalAttention(nn.Module):
    """MQA attention. Local layers: RoPE + sliding-window mask. Global
    layers: no RoPE (NoPE), plain causal attention over everything.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head, use_rope, qk_norm):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.use_rope = use_rope
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)
        self.q_norm = RMSNorm(d_head) if qk_norm else None
        self.k_norm = RMSNorm(d_head) if qk_norm else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.use_rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        # The cache keeps the single KV head (that's MQA's memory saving);
        # expand it to all query heads only for the attention call itself.
        # An explicit copy keeps SDPA on its fast memory-efficient kernel.
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            # Global layer. Training / prefill: square causal. Decode step
            # (one query): may see every cached key -> no mask needed.
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    def __init__(self, cfg, is_global):
        super().__init__()
        self.is_global = is_global
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = LocalGlobalAttention(cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"],
                                         use_rope=not is_global, qk_norm=cfg["qk_norm"])
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg["d_model"], cfg["d_ff"])

    def _forward_inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._forward_inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._forward_inner(x, cos, sin, attn_mask, cache)


class KVCache:
    """Pre-allocated per-layer cache. Keeps the full history (with 1 KV
    head it's only ~0.5 MB per layer at 2048 tokens); the sliding window is
    applied by the attention mask, not by evicting entries.
    """
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


def is_global_layer(layer_idx, global_every):
    return (layer_idx + 1) % global_every == 0


class LocalGlobalGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config, is_global=is_global_layer(i, config["global_every"]))
            for i in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # tied embeddings

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def layer_pattern(self):
        return "".join("G" if b.is_global else "L" for b in self.blocks)

    def new_kv_caches(self, batch_size, device, dtype):
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def _run_blocks(self, x, pos_offset, caches, use_ckpt):
        T = x.shape[1]
        assert pos_offset + T <= self.config["max_seq_len"], "Sequence longer than max_seq_len."
        cos = self.rope_cos[pos_offset:pos_offset + T]
        sin = self.rope_sin[pos_offset:pos_offset + T]
        local_mask = sliding_window_mask(pos_offset, T, pos_offset + T, self.config["window"], x.device,
                                         cacheable=self.training and caches is None)
        for i, blk in enumerate(self.blocks):
            mask = None if blk.is_global else local_mask
            cache = caches[i] if caches is not None else None
            x = blk(x, cos, sin, mask, cache=cache, use_checkpoint=use_ckpt)
        return x

    def encode(self, input_ids, attention_mask=None, mode="causal"):
        """Contract method: hidden states, no LM head. attention_mask is
        accepted for interface compatibility (Phase 1 packs full windows)."""
        assert mode == "causal", "Model B is decoder-only; only causal encoding is supported."
        x = self.embed(input_ids)
        x = self._run_blocks(x, 0, None, use_ckpt=self.training and USE_GRAD_CHECKPOINTING)
        return self.norm_f(x)

    def forward(self, ids, targets=None, caches=None):
        """No targets -> logits. With targets -> tensor [1, 2] = [cross-entropy,
        cross-entropy + z-loss]. The [1, 2] shape is so nn.DataParallel can
        stack one row per GPU; callers take .mean(0).
        In training mode, fp16 autocast is switched on here (inside the
        module) so it also applies inside DataParallel's worker threads.
        """
        use_amp = self.training and ids.is_cuda
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            if caches is None:
                h = self.encode(ids)
            else:
                pos_offset = caches[0].pos
                assert ids.shape[1] == 1 or pos_offset == 0, "Prefill must start from an empty cache."
                h = self.norm_f(self._run_blocks(self.embed(ids), pos_offset, caches, use_ckpt=False))
            logits = self.head(h)
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        target_logit = logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        ce = (lse - target_logit).mean()           # identical to F.cross_entropy
        total = ce + Z_LOSS_COEF * lse.pow(2).mean()
        return torch.stack([ce, total]).unsqueeze(0)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy",
                 top_p=0.9, temperature=1.0, repetition_penalty=1.0):
        """Contract method. Stops per sequence at <eos>. Prompts too long to
        leave room for max_new_tokens are cut to their most recent tokens."""
        assert kv_cache, "Only the KV-cache path is implemented."
        was_training = self.training
        self.eval()
        device, dtype = input_ids.device, next(self.parameters()).dtype

        max_prompt_len = self.config["max_seq_len"] - max_new_tokens
        if input_ids.shape[1] > max_prompt_len:
            input_ids = input_ids[:, -max_prompt_len:]

        B, T0 = input_ids.shape
        max_total = min(T0 + max_new_tokens, self.config["max_seq_len"])
        caches = self.new_kv_caches(batch_size=B, device=device, dtype=dtype)
        generated = input_ids
        done = torch.zeros(B, dtype=torch.bool, device=device)

        logits = self.forward(input_ids, caches=caches)[:, -1, :]
        next_ids = self._sample(self._penalize(logits, generated, repetition_penalty), sampler, top_p, temperature)
        while generated.shape[1] < max_total:
            next_ids = torch.where(done, torch.full_like(next_ids, EOS_ID), next_ids)
            generated = torch.cat([generated, next_ids.unsqueeze(1)], dim=1)
            done = done | (next_ids == EOS_ID)
            if done.all() or generated.shape[1] >= max_total:
                break
            logits = self.forward(next_ids.unsqueeze(1), caches=caches)[:, -1, :]
            next_ids = self._sample(self._penalize(logits, generated, repetition_penalty), sampler, top_p, temperature)

        self.train(was_training)
        return generated

    @staticmethod
    def _penalize(logits, previous_ids, penalty):
        """CTRL-style repetition penalty on every token already present."""
        if penalty == 1.0:
            return logits
        scores = logits.gather(1, previous_ids)
        scores = torch.where(scores > 0, scores / penalty, scores * penalty)
        return logits.scatter(1, previous_ids, scores)

    @staticmethod
    def _sample(logits, sampler, top_p, temperature):
        if sampler == "greedy":
            return logits.argmax(dim=-1)
        probs = F.softmax(logits / temperature, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
        cumulative = sorted_probs.cumsum(dim=-1)
        cutoff = (cumulative <= top_p).sum(dim=-1, keepdim=True).clamp(min=1)
        keep = torch.arange(sorted_probs.shape[-1], device=logits.device)[None, :] < cutoff
        sorted_probs = sorted_probs * keep
        sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
        picked = torch.multinomial(sorted_probs, num_samples=1).squeeze(-1)
        return sorted_idx.gather(-1, picked.unsqueeze(-1)).squeeze(-1)


def build_model(config):
    """Contract function."""
    return LocalGlobalGPT(config)


USE_GRAD_CHECKPOINTING = True

In [ ]:
%%writefile /kaggle/working/sum_lib.py
"""sum_lib.py -- Phase 6 (abstractive legal summarisation) library.

Problem: judgments are thousands of words, every Phase-1 model sees at most ~1,024 tokens, and the
metric (BERTScore F1, roberta-large) only reads the first ~510 tokens of each summary.

Pipeline (every experiment) -- "plan, then write section by section":
  SEGMENT   each judgment is cut into consecutive ~SEG_WORDS-word segments (sentence boundaries).
  ALIGN     each sentence of the reference summary is assigned to the segment that best supports it
            (IDF-weighted word recall, near-monotone dynamic programme). This turns one long
            (document, summary) pair into many short (segment, sub-summary) pairs that fit the
            models (divide-and-conquer training, Gidiotis & Tsoumakas 2020).
  PLAN      a length model learned on train: total summary words ~ power law of document words, spread
            over segments by a learned positional profile (which parts of a judgment summaries
            draw from). Segments whose share is under MIN_WORDS are skipped (content selection).
  WRITE     the adapted Phase-1 model writes each planned segment's part, conditioned on a short cue
            "[<position> part] Summary (about N words):" (length + position control, Fan et al.
            2018); the parts are joined in document order.
  TUNE      the length scale (0.8 / 1.0 / 1.25 x plan) is chosen on held-out tune documents by BERTScore.

Adaptation methods (own code): LoRMA (Bihany et al. 2025), LoRA (Hu et al. 2021), QLoRA (Dettmers et
al. 2023: NF4 4-bit frozen base + double-quantised scales + LoRA), Houlsby adapters (2019), prefix
tuning (Li & Liang 2021; deep, encoder side). Distillation: half-depth D student, word-level KD from the
LoRMA teacher's token distributions (Hinton et al. 2015; Kim & Rush 2016).
"""
import json
import math
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEG_WORDS = 450                 # words per document segment
TGT_MAX_TOK = 256               # longest sub-summary the model writes in one go
MIN_WORDS = 12                  # planned parts shorter than this are skipped
LEN_SCALES = (0.8, 1.0, 1.25)
POS_WORDS = ("beginning", "early", "middle", "late", "end")
CUE_NS = tuple(range(10, 410, 10))
N_TUNE, N_DEV = 40, 80
LINEAR_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
ADAPTER_TARGETS = ("o_proj", "down_proj")


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
def load_json_any(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError:
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]


def _first(d, keys, default=None):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default


def parse_docs(obj):
    """-> list of {"id": str, "raw_id", "sents": [str] or None, "text": str, "summary": str or None}.
    'document' may be a string or a list of sentences."""
    if isinstance(obj, dict):
        for k in ("data", "documents", "docs", "examples", "items"):
            if k in obj and isinstance(obj[k], list):
                return parse_docs(obj[k])
        return parse_docs([dict(v, id=v.get("id", k)) if isinstance(v, dict) else {"id": k, "document": v} for k, v in obj.items()])
    out = []
    for i, v in enumerate(obj):
        raw = _first(v, ("id", "doc_id", "case_id", "name"), i)
        doc = _first(v, ("document", "text", "judgment", "judgement", "source", "doc"), "")
        sents = [" ".join(str(s).split()) for s in doc if str(s).strip()] if isinstance(doc, list) else None
        text = " ".join(sents) if sents is not None else " ".join(str(doc).split())
        summ = _first(v, ("summary", "abstract", "headnote", "target", "reference"))
        if isinstance(summ, list):
            summ = " ".join(str(s) for s in summ)
        out.append({"id": str(raw), "raw_id": raw, "sents": sents, "text": text,
                    "summary": None if summ is None else " ".join(str(summ).split())})
    return out


def _h(t):
    import hashlib
    return hashlib.md5(t.encode("utf-8")).hexdigest()


def clean_and_split(train, test, seed=7, log=print):
    """Test ids must be unique (exact repeats dropped). Train: documents also in test dropped,
    exact duplicates dropped, empty documents / summaries dropped. Then train -> train / tune / dev
    (held-out documents: tune picks the length scale, dev is for the report)."""
    rep = {"test_rows": len(test), "train_rows": len(train)}
    seen, t2 = set(), []
    for d in test:
        if d["id"] in seen:
            continue
        seen.add(d["id"])
        t2.append(d)
    rep["test_repeats_dropped"] = len(test) - len(t2)
    held = {_h(d["text"]) for d in t2}
    keep, seen = [], set()
    rep["train_leak_dropped"] = rep["train_dup_dropped"] = rep["train_empty_dropped"] = 0
    for d in train:
        k = _h(d["text"])
        if not d["text"] or not d["summary"]:
            rep["train_empty_dropped"] += 1
        elif k in held:
            rep["train_leak_dropped"] += 1
        elif k in seen:
            rep["train_dup_dropped"] += 1
        else:
            seen.add(k)
            keep.append(d)
    ids, uniq = Counter(), []
    for d in keep:                                  # unique internal ids
        ids[d["id"]] += 1
        uniq.append(dict(d, id=f"{d['id']}#{ids[d['id']]}") if ids[d["id"]] > 1 else d)
    rng = random.Random(seed)
    rng.shuffle(uniq)
    n_tune = min(N_TUNE, max(5, len(uniq) // 20))
    n_dev = min(N_DEV, max(5, len(uniq) // 10))
    split = {"tune": uniq[:n_tune], "dev": uniq[n_tune:n_tune + n_dev], "train": uniq[n_tune + n_dev:], "test": t2}
    for s in split:
        for d in split[s]:
            d["id"] = f"{s}:{d['id']}"
    rep.update({f"{s}_docs": len(v) for s, v in split.items()})
    return split, rep


# ---------------------------------------------------------------------------
# Text: sentences, segments, alignment, plan
# ---------------------------------------------------------------------------
_ABBR = {"no", "nos", "v", "vs", "ltd", "co", "sec", "s", "ss", "art", "arts", "mr", "mrs", "ms", "dr", "hon'ble", "honble", "i.e",
         "e.g", "etc", "viz", "cl", "para", "paras", "pvt", "inc", "govt", "j", "jj", "u/s", "r/w", "ors", "anr", "st", "sh", "smt", "p", "pp",
         "vol", "ed", "cr", "crl", "civ", "w.p", "s.l.p", "a.i.r", "air", "scc", "scr", "supp"}
_SPLIT = re.compile(r"(?<=[.!?;])\s+(?=[\"'(\[]?[A-Z0-9])")


def split_sentences(text):
    parts, out = _SPLIT.split(text), []
    for p in parts:
        if out:
            last = out[-1].split()[-1].lower().rstrip(".") if out[-1].split() else ""
            if last in _ABBR or (len(last) == 1 and last.isalpha()):
                out[-1] += " " + p
                continue
        out.append(p)
    return [s.strip() for s in out if s.strip()]


def segment_doc(sents, seg_words=SEG_WORDS):
    """Consecutive sentences packed into segments of <= seg_words words (long sentences are cut)."""
    segs, cur, n = [], [], 0
    for s in sents:
        w = s.split()
        while len(w) > seg_words:                   # a very long "sentence" (lists, tables)
            if cur:
                segs.append(" ".join(cur))
                cur, n = [], 0
            segs.append(" ".join(w[:seg_words]))
            w = w[seg_words:]
        if n + len(w) > seg_words and cur:
            segs.append(" ".join(cur))
            cur, n = [], 0
        if w:
            cur.append(" ".join(w))
            n += len(w)
    if cur:
        segs.append(" ".join(cur))
    return segs or [""]


_WORD = re.compile(r"[a-z0-9]+")
STOP = set("the of and to in a is that be by for on as was with it this which or are at an not his her their from have has had "
           "been were said under shall may any such other these those there would could should will also no its upon all "
           "into than so if but we they he she it them him who whom whose being per".split())


def content_words(text):
    return [w for w in _WORD.findall(text.lower()) if w not in STOP and len(w) > 1]


def align(summary_sents, segments, idf, back_penalty=0.1):
    """Assign every summary sentence to one segment: maximise sum of IDF-weighted recall of the
    sentence's content words in the segment, paying back_penalty each time the order goes backwards
    (summaries mostly follow the judgment). Returns (assignment list, per-sentence scores)."""
    K, n = len(segments), len(summary_sents)
    if n == 0:
        return [], []
    segw = [set(content_words(s)) for s in segments]
    S = np.zeros((n, K))
    for j, s in enumerate(summary_sents):
        w = set(content_words(s))
        tot = sum(idf.get(x, 1.0) for x in w)
        if tot:
            for k in range(K):
                S[j, k] = sum(idf.get(x, 1.0) for x in w & segw[k]) / tot
    best = S[0].copy()
    back = np.zeros((n, K), dtype=int)
    for j in range(1, n):
        pre = np.maximum.accumulate(best)                          # best over k' <= k (forward, free)
        pre_arg = np.array([int(np.argmax(best[:k + 1])) for k in range(K)])
        new = np.empty(K)
        for k in range(K):
            b_back = best[k + 1:].max() - back_penalty if k + 1 < K else -1e9
            if pre[k] >= b_back:
                new[k], back[j, k] = pre[k], pre_arg[k]
            else:
                new[k], back[j, k] = b_back, k + 1 + int(np.argmax(best[k + 1:]))
        best = new + S[j]
    a = [int(np.argmax(best))]
    for j in range(n - 1, 0, -1):
        a.append(int(back[j, a[-1]]))
    a = a[::-1]
    return a, [float(S[j, a[j]]) for j in range(n)]


def build_idf(docs):
    df = Counter()
    for d in docs:
        df.update(set(content_words(d["text"])))
    n = len(docs)
    return {w: math.log((n + 1) / (c + 1)) + 1 for w, c in df.items()}


def pos_bucket(frac):
    return POS_WORDS[min(4, int(frac * 5))]


def doc_segments(d):
    sents = d["sents"] if d["sents"] is not None else split_sentences(d["text"])
    segs = segment_doc(sents)
    words = [len(s.split()) for s in segs]
    tot = max(1, sum(words))
    cum, info = 0, []
    for s, w in zip(segs, words):
        info.append({"text": s, "words": w, "centre": (cum + w / 2) / tot})
        cum += w
    return info


def make_pairs(docs, idf):
    """(segment, aligned sub-summary) training pairs + statistics for the plan."""
    pairs, prof_num, prof_den, lens, scores = [], np.zeros(10), np.zeros(10), [], []
    for d in docs:
        segs = doc_segments(d)
        ss = split_sentences(d["summary"])
        a, sc = align(ss, [s["text"] for s in segs], idf)
        scores += sc
        parts = [[] for _ in segs]
        for j, k in enumerate(a):
            parts[k].append(ss[j])
        lens.append((sum(s["words"] for s in segs), len(d["summary"].split())))
        for k, (s, p) in enumerate(zip(segs, parts)):
            sub = " ".join(p)
            dec = min(9, int(s["centre"] * 10))
            prof_num[dec] += len(sub.split())
            prof_den[dec] += s["words"]
            if sub:
                pairs.append({"doc": d["id"], "k": k, "seg": s["text"], "pos": pos_bucket(s["centre"]), "tgt": sub,
                              "n": len(sub.split())})
    return pairs, {"profile": (prof_num / np.maximum(prof_den, 1)).tolist(), "lengths": lens,
                   "align_scores": scores}


def fit_length_model(lengths):
    """log(summary words) = a + b log(document words), least squares; plus clipping range."""
    x = np.log([max(1, d) for d, _ in lengths])
    y = np.log([max(1, s) for _, s in lengths])
    b, a = np.polyfit(x, y, 1) if len(set(x)) > 1 else (0.0, float(np.mean(y)))
    s = np.array([s for _, s in lengths])
    return {"a": float(a), "b": float(b), "lo": float(np.quantile(s, 0.05)), "hi": float(np.quantile(s, 0.95))}


def plan_doc(d, lm, profile):
    """Per-segment word budgets: total from the length model, shared out by segment size x the
    positional profile; parts under MIN_WORDS are dropped (the best one is always kept)."""
    segs = doc_segments(d)
    W = sum(s["words"] for s in segs)
    total = float(np.clip(math.exp(lm["a"] + lm["b"] * math.log(max(1, W))), lm["lo"], lm["hi"]))
    raw = np.array([s["words"] * max(1e-3, profile[min(9, int(s["centre"] * 10))]) for s in segs], dtype=float)
    if raw.sum() <= 0:                              # empty document
        raw = np.ones(len(segs))
    cap = TGT_MAX_TOK / 1.7
    bud = np.minimum(raw / raw.sum() * total, cap)
    keep = [k for k in range(len(segs)) if bud[k] >= MIN_WORDS] or [int(np.argmax(bud))]
    for _ in range(3):                              # give the words of skipped parts to the kept ones
        kb = bud[keep]
        if kb.sum() <= 0 or abs(kb.sum() - total) < 1:
            break
        bud[keep] = np.minimum(kb * total / kb.sum(), cap)
    for k, s in enumerate(segs):
        s["budget"] = float(bud[k]) if k in keep else float(min(bud[k], MIN_WORDS - 1e-6))
        s["pos"] = pos_bucket(s["centre"])
    return {"id": d["id"], "raw_id": d["raw_id"], "summary": d.get("summary"), "text": d["text"], "doc_words": W, "planned_words": total,
            "segments": segs, "write": keep}


def cue_text(pos, n):
    return f"\n[{pos} part] Summary (about {n} words):"


def cue_n(words):
    return int(min(CUE_NS[-1], max(CUE_NS[0], round(words / 10) * 10)))


# ---------------------------------------------------------------------------
# Metrics (own ROUGE; BERTScore through the bert-score package, roberta-large, as the evaluator)
# ---------------------------------------------------------------------------
def _toks(t):
    return re.findall(r"[a-z0-9]+", t.lower())


def _lcs(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        x = masks.get(t, 0) | state
        state = x & ~(x - ((state << 1) | 1))
    return bin(state).count("1")


def rouge(pred, ref):
    p, r = _toks(pred), _toks(ref)
    if not p or not r:
        return 0.0, 0.0
    cp, cr = Counter(p), Counter(r)
    o = sum((cp & cr).values())
    r1 = 2 * o / (len(p) + len(r))
    l = _lcs(p, r)
    rl = 2 * l / (len(p) + len(r))
    return r1, rl


def text_stats(pred, src, ref=None):
    p = _toks(pred)
    tri = [tuple(p[i:i + 3]) for i in range(len(p) - 2)]
    red = 1 - len(set(tri)) / max(1, len(tri))
    s = set(_toks(src))
    s4 = {tuple(_toks(src)[i:i + 4]) for i in range(max(0, len(_toks(src)) - 3))}
    p4 = [tuple(p[i:i + 4]) for i in range(len(p) - 3)]
    nums = [w for w in p if w.isdigit()]
    out = {"words": len(pred.split()), "repeated_trigrams": red, "novel_4grams": sum(g not in s4 for g in p4) / max(1, len(p4)),
           "numbers_in_source": sum(n in s for n in nums) / max(1, len(nums))}
    if ref is not None:
        rw = set(content_words(ref))
        out["ref_content_covered"] = len(rw & set(content_words(pred))) / max(1, len(rw))
    return out


_SCORER = {}


def bertscore(preds, refs, device="cuda"):
    """BERTScore F1 with the evaluator's settings (bert_score defaults for lang='en': roberta-large,
    layer 17, no idf, no rescaling); the scorer is loaded once per process. None if the package /
    weights are not available (then ROUGE-1 is used for choices). Local scoring only."""
    try:
        import bert_score
        dev = device if torch.cuda.is_available() else "cpu"
        if dev not in _SCORER:
            _SCORER[dev] = bert_score.BERTScorer(lang="en", device=dev)
        _, _, f1 = _SCORER[dev].score([p or "." for p in preds], [r or "." for r in refs], batch_size=8, verbose=False)
        return [float(x) for x in f1]
    except Exception as e:                      # noqa: BLE001
        print(f"  (BERTScore unavailable: {type(e).__name__}: {str(e)[:120]})", flush=True)
        return None


def evaluate(preds, docs, device="cuda", with_bert=True):
    refs = [d["summary"] for d in docs]
    r = [rouge(p, t) for p, t in zip(preds, refs)]
    bs = bertscore(preds, refs, device) if with_bert else None
    st = [text_stats(p, d["text"] if "text" in d else "", d["summary"]) for p, d in zip(preds, docs)]
    out = {"docs": len(preds), "rouge1": round(float(np.mean([x[0] for x in r])), 4), "rougeL": round(float(np.mean([x[1] for x in r])), 4),
           "bertscore_f1": None if bs is None else round(float(np.mean(bs)), 4)}
    for k in st[0]:
        out[k] = round(float(np.mean([s[k] for s in st])), 4)
    out["ref_words"] = round(float(np.mean([len(t.split()) for t in refs])), 1)
    return out, bs


def primary(m):
    return m["bertscore_f1"] if m["bertscore_f1"] is not None else m["rouge1"]


# ---------------------------------------------------------------------------
# Adaptation methods
# ---------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """LoRA: y = W0 x + s B A x, B = 0 at the start (exactly the Phase-1 layer)."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    """LoRMA, additive-identity form: y = (I + s B A) W0 x = h + s B A h with h = W0 x.
    B = 0 at the start, so the layer starts exactly as the Phase-1 layer."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A), self.B) * self.scale


NF4_VALUES = (-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
              -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
              0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
              0.7229568362236023, 1.0)


def quantize_nf4(w, block=64, dq_block=256):
    """4-bit NormalFloat (QLoRA): blocks of `block` weights scaled by their absmax, each weight
    -> nearest of 16 normal quantiles (2 codes per byte). Double quantisation: the block absmax
    values are themselves stored as int8 (minus their mean, scaled per `dq_block` blocks)."""
    flat = w.detach().float().flatten()
    n = flat.numel()
    flat = F.pad(flat, (0, (-n) % (2 * block)))
    blocks = flat.view(-1, block)
    absmax = blocks.abs().amax(1).clamp(min=1e-12)
    table = torch.tensor(NF4_VALUES)
    idx = torch.bucketize((blocks / absmax[:, None]).flatten(), (table[1:] + table[:-1]) / 2)
    codes = ((idx[0::2] << 4) | idx[1::2]).to(torch.uint8)
    c = absmax.mean()
    a = absmax - c
    nb = a.numel()
    a = F.pad(a, (0, (-nb) % dq_block)).view(-1, dq_block)
    s2 = a.abs().amax(1).clamp(min=1e-12)
    qabs = torch.round(a / s2[:, None] * 127).clamp(-127, 127).to(torch.int8).flatten()
    return {"codes": codes, "qabs": qabs, "s2": s2.float(), "c": c.float().reshape(1), "n": n, "nb": nb,
            "shape": tuple(w.shape), "block": block, "dq_block": dq_block}


def dequantize_nf4(codes, qabs, s2, c, table, n, nb, shape, block, dq_block, dtype=torch.float32):
    idx = torch.stack([(codes >> 4).long(), (codes & 15).long()], 1).flatten()
    absmax = ((qabs.view(-1, dq_block).float() / 127) * s2[:, None]).flatten()[:nb] + c
    w = (table[idx].view(-1, block).float() * absmax[:, None]).flatten()[:n]
    return w.view(shape).to(dtype)


class QLoRALinear(nn.Module):
    """QLoRA: frozen base weight stored in NF4 (4-bit) + double-quantised scales, de-quantised on
    the fly; trainable LoRA y = W_nf4 x + s B A x (B = 0, so it starts as the 4-bit model)."""

    def __init__(self, base, r, alpha, block=64, dq_block=256):
        super().__init__()
        q = quantize_nf4(base.weight, block, dq_block)
        self.out_features, self.in_features = base.out_features, base.in_features
        for k in ("codes", "qabs", "s2", "c"):
            self.register_buffer(k, q[k])
        self.register_buffer("table", torch.tensor(NF4_VALUES))
        self.meta = (q["n"], q["nb"], q["shape"], block, dq_block)
        with torch.no_grad():                       # how far the 4-bit weight is from the Phase-1 weight
            w = base.weight.detach().float()
            self._err = float((self.weight() - w).norm() / w.norm().clamp(min=1e-12))
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def weight(self, dtype=torch.float32):
        return dequantize_nf4(self.codes, self.qabs, self.s2, self.c, self.table, *self.meta, dtype=dtype)

    def forward(self, x):
        dt = torch.float16 if x.is_cuda else torch.float32
        if self.training:                           # training: de-quantise on the fly (only 4-bit kept)
            self._wcache = None
            W = self.weight(dt)
        else:                                       # generation: de-quantise once, reuse at every step
            if getattr(self, "_wcache", None) is None or self._wcache.dtype != dt or self._wcache.device != x.device:
                self._wcache = self.weight(dt)
            W = self._wcache
        y = F.linear(x, W, self.bias)
        return y + F.linear(F.linear(x, self.A), self.B) * self.scale


class AdapterLinear(nn.Module):
    """Houlsby adapter on a sublayer output (before the residual add): y = h + U gelu(D h), h = W0 x.
    U = 0 at the start, so the layer starts exactly as the Phase-1 layer."""

    def __init__(self, base, m):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.out_features, m)
        self.up = nn.Linear(m, base.out_features)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.up(F.gelu(self.down(h)))


class PrefixTuner(nn.Module):
    """Prefix tuning, deep form (Li & Liang 2021; P-Tuning v2): P extra positions at the front of
    every chunk whose hidden state at the INPUT OF EVERY BLOCK is a learned vector. Every real token
    can attend to them at every layer (their keys/values come from the learned states through the
    frozen projections). Implemented with forward pre-hooks, so no model code changes."""

    def __init__(self, blocks, n_prefix, d):
        super().__init__()
        self.P = n_prefix
        self.prefix = nn.Parameter(torch.zeros(len(blocks), n_prefix, d))
        self.capture = None
        self.handles = [b.register_forward_pre_hook(self._hook(i)) for i, b in enumerate(blocks)]

    def _hook(self, i):
        def hook(module, args):
            x = args[0]
            if self.capture is not None:            # initialisation pass: record real activations
                v = x[:, self.P:2 * self.P].detach().float().mean(0)
                if v.shape[0] < self.P:
                    v = torch.cat([v, v.mean(0, keepdim=True).expand(self.P - v.shape[0], -1)])
                self.capture[i] = v
                return None
            p = self.prefix[i].to(x.dtype).unsqueeze(0).expand(x.shape[0], -1, -1)
            return (torch.cat([p, x[:, self.P:]], 1),) + tuple(args[1:])
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()


def _replace(module, name, new):
    parent = module
    parts = name.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new)


def norm_params(model):
    return [p for m in model.modules() if type(m).__name__.endswith("Norm") for p in m.parameters(recurse=False)]

def storage_mb(model):
    """Deployable backbone size: fp16 parameters + packed 4-bit buffers (QLoRA)."""
    b = sum(p.numel() * 2 for p in model.parameters())
    b += sum(buf.numel() * buf.element_size() for n, buf in model.named_buffers() if n.split(".")[-1] in ("codes", "qabs", "s2", "c"))
    return round(b / 2 ** 20, 1)


# ---------------------------------------------------------------------------


def blocks_of(kind, model):
    return model.enc_blocks if kind in ("D", "Ds") else model.blocks


def apply_method(model, kind, method, exp):
    """Freeze the Phase-1 model, then insert / open what the method trains.
    Returns (trainable backbone parameters, PrefixTuner or None)."""
    for p in model.parameters():
        p.requires_grad_(False)
    tuner = None
    if method == "full":
        for p in model.parameters():
            p.requires_grad_(True)
    elif method in ("lorma", "lora", "qlora", "adapter"):
        targets = ADAPTER_TARGETS if method == "adapter" else LINEAR_TARGETS
        names = [n for n, m in model.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in targets]
        assert names, "no target linear layers found"
        for n in names:
            lin = model.get_submodule(n)
            if method == "lorma":
                new = LoRMALinear(lin, exp.get("r", 16), exp.get("alpha", 32))
            elif method == "lora":
                new = LoRALinear(lin, exp.get("r", 16), exp.get("alpha", 32))
            elif method == "qlora":
                new = QLoRALinear(lin, exp.get("r", 16), exp.get("alpha", 32))
            else:
                new = AdapterLinear(lin, exp.get("bottleneck", 64))
            _replace(model, n, new)
        if method == "adapter":                    # Houlsby et al. also train the layer norms
            for p in norm_params(model):
                p.requires_grad_(True)
    elif method == "prefix":                       # encoder side for D (the decoder reads it by cross-attention)
        tuner = PrefixTuner(list(blocks_of(kind, model)), exp.get("n_prefix", 16), exp["d_model"])
    else:
        raise ValueError(method)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return n + (tuner.prefix.numel() if tuner is not None else 0), tuner


def family(kind):
    return "D" if kind in ("D", "Ds") else ("C" if kind == "C" else "AB")


def limits(kind, model):
    """(max prompt tokens, max new tokens) that fit the model's context."""
    cfg = model.config
    if family(kind) == "D":
        return cfg["max_enc_len"], min(TGT_MAX_TOK, cfg["max_dec_len"] - 1)
    return cfg["max_seq_len"] - TGT_MAX_TOK - 1, TGT_MAX_TOK


def head_weight(model):
    return model.head.weight


def amp_ctx(device):
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda")


def _pad(seqs, pad, device, lead=0):
    L = max(len(s) for s in seqs) + lead
    ids = torch.full((len(seqs), L), pad, dtype=torch.long)
    mask = torch.zeros(len(seqs), L, dtype=torch.bool)
    for b, s in enumerate(seqs):
        ids[b, lead:lead + len(s)] = torch.as_tensor(np.asarray(s, dtype=np.int64))
        mask[b, :lead + len(s)] = True
    return ids.to(device), mask.to(device)


def target_logits(kind, model, prompts, targets, pad, start_id, device, n_prefix=0):
    """Teacher-forced logits at every target position -> (logits [N, V] fp32, labels [N])."""
    fam = family(kind)
    with amp_ctx(device):
        if fam == "D":
            enc_ids, enc_mask = _pad(prompts, pad, device, lead=n_prefix)
            T = max(len(t) for t in targets)
            dec = torch.full((len(targets), T), pad, dtype=torch.long)
            lab = torch.full((len(targets), T), -100, dtype=torch.long)
            for b, t in enumerate(targets):
                t = torch.as_tensor(np.asarray(t, dtype=np.int64))
                dec[b, 0] = start_id
                dec[b, 1:len(t)] = t[:-1]
                lab[b, :len(t)] = t
            dec, lab = dec.to(device), lab.to(device)
            h = model.decode_hidden(dec, model.encode(enc_ids, enc_mask), enc_mask)
            sel = lab != -100
            hs, y = h[sel], lab[sel]
        else:
            seqs = [list(p) + list(t) for p, t in zip(prompts, targets)]
            ids, mask = _pad(seqs, pad, device)
            if fam == "C":
                plen = torch.tensor([len(p) for p in prompts], device=device)
                h = model.encode(ids, mode="prefix", prefix_len=plen)
            else:
                h = model.encode(ids)
            rows, cols, ys = [], [], []
            for b, (p, t) in enumerate(zip(prompts, targets)):
                rows += [b] * len(t)
                cols += list(range(len(p) - 1, len(p) - 1 + len(t)))
                ys += [int(x) for x in t]
            hs = h[torch.tensor(rows, device=device), torch.tensor(cols, device=device)]
            y = torch.tensor(ys, device=device)
        logits = F.linear(hs, head_weight(model))
    return logits.float(), y


# ---------------------------------------------------------------------------
# Generation (greedy, KV cache, EOS stop, 4-gram blocking, repetition penalty)
# ---------------------------------------------------------------------------
def _decode_loop(first_logits, step, B, max_new, eos, no_repeat, rep_penalty, vocab_limit=None, banned=()):
    logits = first_logits
    gen, done, seen = [[] for _ in range(B)], [False] * B, [dict() for _ in range(B)]
    used = torch.zeros(B, first_logits.shape[-1], dtype=torch.bool, device=first_logits.device)
    for t in range(max(max_new)):
        logits = logits.float()
        if vocab_limit is not None and vocab_limit < logits.shape[-1]:
            logits[:, vocab_limit:] = -float("inf")        # ids the tokenizer cannot decode
        if banned:
            logits[:, list(banned)] = -float("inf")        # padding / unknown / start tokens
        if rep_penalty != 1.0 and t > 0:           # CTRL-style penalty on tokens already written (one op)
            logits = torch.where(used, torch.where(logits > 0, logits / rep_penalty, logits * rep_penalty), logits)
        if no_repeat:
            for b in range(B):
                if len(gen[b]) >= no_repeat - 1:
                    bad = seen[b].get(tuple(gen[b][-(no_repeat - 1):]))
                    if bad:
                        logits[b, list(bad)] = -float("inf")
        nxt_t = logits.argmax(-1)
        used[torch.arange(B, device=used.device), nxt_t] = True
        nxt = nxt_t.tolist()
        for b in range(B):
            if done[b]:
                nxt[b] = eos
                continue
            if nxt[b] == eos:
                done[b] = True
                continue
            gen[b].append(nxt[b])
            if no_repeat and len(gen[b]) >= no_repeat:
                g = gen[b][-no_repeat:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
            if len(gen[b]) >= max_new[b]:
                done[b] = True
        if all(done):
            break
        logits = step(torch.tensor(nxt, device=first_logits.device), t)
    return gen


@torch.no_grad()
def _gen_batch(kind, model, prompts, max_new, eos, pad, start_id, device, n_prefix, no_repeat, rep_penalty, vocab_limit=None, banned=()):
    import importlib
    fam = family(kind)
    B, Tn = len(prompts), max(max_new)
    with amp_ctx(device):
        cdt = torch.float16 if device.type == "cuda" else torch.float32
        if fam == "D":
            import legal_d
            enc_ids, enc_mask = _pad(prompts, pad, device, lead=n_prefix)
            enc_out = model.encode(enc_ids, enc_mask)
            kvs = [blk.cross_attn.project_kv(enc_out) for blk in model.dec_blocks]
            caches = [legal_d.KVCache(Tn + 1) for _ in model.dec_blocks]
            tok = torch.full((B, 1), start_id, dtype=torch.long, device=device)
            first = model.decoder_step(tok, 0, caches, kvs, enc_mask)[:, -1]

            def step(nxt, t):
                with amp_ctx(device):
                    return model.decoder_step(nxt[:, None], t + 1, caches, kvs, enc_mask)[:, -1]
        elif fam == "C":
            cm = importlib.import_module("c_model")
            cfg = model.config
            T0 = max(len(p) for p in prompts)
            total = T0 + Tn + 1
            pads = torch.tensor([T0 - len(p) for p in prompts], device=device)
            ids = torch.full((B, T0), pad, dtype=torch.long, device=device)
            for b, p in enumerate(prompts):
                ids[b, T0 - len(p):] = torch.as_tensor(np.asarray(p, dtype=np.int64), device=device)
            idx = torch.arange(total, device=device)
            key_ok = idx[None, :] >= pads[:, None]
            positions = (idx[None, :] - pads[:, None]).clamp(min=0)
            caches = [cm.KVCache(B, cfg["n_kv_head"], total, cfg["d_head"], device, cdt) for _ in model.blocks]
            mask0 = (key_ok[:, None, :T0] | torch.eye(T0, dtype=torch.bool, device=device)[None])[:, None]

            def run(x_ids, pos, mask):
                x = model.embed(x_ids)
                cos, sin = model.rope_cos[pos], model.rope_sin[pos]
                for blk, cache in zip(model.blocks, caches):
                    x = blk(x, cos, sin, mask, cache=cache)
                return model.head(model.norm_f(x[:, -1:]))[:, -1]
            first = run(ids, positions[:, :T0], mask0)

            def step(nxt, t):
                cur = T0 + t
                with amp_ctx(device):
                    return run(nxt[:, None], positions[:, cur:cur + 1], key_ok[:, None, None, :cur + 1])
        else:                                       # A / B: prompts in a batch have equal length (see generate)
            ids = torch.as_tensor(np.asarray(prompts, dtype=np.int64), device=device)
            caches = model.new_kv_caches(B, device, cdt)

            def run(x_ids):
                if hasattr(model, "_run_blocks"):                                   # B
                    h = model._run_blocks(model.embed(x_ids), caches[0].pos, caches, use_ckpt=False)
                else:                                                               # A
                    pos, T = caches[0].pos, x_ids.shape[1]
                    h = model.embed(x_ids)
                    cos, sin = model.rope_cos[pos:pos + T], model.rope_sin[pos:pos + T]
                    for i, blk in enumerate(model.blocks):
                        h = blk(h, cos, sin, cache=caches[i], use_checkpoint=False)
                return model.head(model.norm_f(h[:, -1:]))[:, -1]
            first = run(ids)

            def step(nxt, t):
                with amp_ctx(device):
                    return run(nxt[:, None])
    return _decode_loop(first, step, B, max_new, eos, no_repeat, rep_penalty, vocab_limit, banned)


def generate(kind, model, prompts, max_new, eos, pad, start_id, device, bs, n_prefix=0, no_repeat=4, rep_penalty=1.15,
             vocab_limit=None, banned=()):
    """Prompts sorted by length and batched. A / B have no padding mask, so every prompt in a batch
    is cut (from the front of its segment) to the batch's shortest length -- with length-sorted
    batches this drops at most a few tokens."""
    model.eval()
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    out = [None] * len(prompts)
    for s in range(0, len(order), bs):
        part = order[s:s + bs]
        ps = [list(prompts[i]) for i in part]
        if family(kind) == "AB":
            m = min(len(p) for p in ps)
            ps = [p[len(p) - m:] for p in ps]
        res = _gen_batch(kind, model, ps, [max_new[i] for i in part], eos, pad, start_id, device, n_prefix, no_repeat, rep_penalty,
                         vocab_limit, banned)
        for i, g in zip(part, res):
            out[i] = g
    return out


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def lr_mult(p, warm=0.05):
    return min(1.0, p / warm) if p < warm else max(0.1, 1 - (p - warm) / (1 - warm))


def make_prompt(seg_ids, cue_ids, cap):
    keep = max(1, cap - len(cue_ids))
    return list(seg_ids[:keep]) + list(cue_ids)


def batches(pairs, mb, rng, bucket=50):
    """Length-bucketed shuffled micro-batches (little padding, random order)."""
    idx = list(range(len(pairs)))
    rng.shuffle(idx)
    out = []
    for s in range(0, len(idx), mb * bucket):
        block = sorted(idx[s:s + mb * bucket], key=lambda i: len(pairs[i]["seg"]) + len(pairs[i]["tgt"]))
        out += [block[k:k + mb] for k in range(0, len(block), mb)]
    rng.shuffle(out)
    return out


def init_prefix(kind, model, tuner, prompts, pad, device):
    """Li & Liang: start the prefix from activations of real words (first P tokens of real segments)."""
    if tuner is None:
        return
    model.eval()
    tuner.capture = {}
    with torch.no_grad(), amp_ctx(device):
        ids, mask = _pad(prompts, pad, device, lead=tuner.P)
        model.encode(ids, mask) if family(kind) == "D" else model.encode(ids)
    for i, v in tuner.capture.items():
        tuner.prefix.data[i] = v.to(tuner.prefix.dtype)
    tuner.capture = None


@torch.no_grad()
def pairs_nll(kind, model, pairs, data, cap, device, bs, n_prefix=0):
    model.eval()
    tot, n = 0.0, 0
    for s in range(0, len(pairs), bs):
        part = pairs[s:s + bs]
        lg, y = target_logits(kind, model, [make_prompt(p["seg"], p["cue"], cap) for p in part], [p["tgt"] for p in part],
                              data["pad"], data["start"], device, n_prefix)
        tot += float(F.cross_entropy(lg, y, reduction="sum"))
        n += len(y)
    return tot / max(1, n)


def train(exp, data, model, kind, tuner, device, log, teacher=None):
    rng = random.Random(exp["seed"])
    torch.manual_seed(exp["seed"])
    cap, _ = limits(kind, model)
    P = tuner.P if tuner is not None else 0
    cap -= P
    tr, va = data["pairs"]["train"], data["pairs"]["tune"][: exp.get("val_pairs", 400)]
    params = [p for p in model.parameters() if p.requires_grad] + ([tuner.prefix] if tuner is not None else [])
    opt = torch.optim.AdamW([{"params": params, "lr": exp["lr"], "base_lr": exp["lr"],
                              "weight_decay": 0.0 if tuner is not None else 0.01}])
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    mb, accum = exp["micro_bs"], exp["accum"]
    planned = max(1, exp["epochs"] * math.ceil(len(tr) / mb) // accum)
    budget = exp["budget_min"] * 60
    T, alpha = exp.get("kd_T", 2.0), exp.get("kd_alpha", 0.0)
    t0, step, micro, progress = time.time(), 0, 0, 0.0
    st = {"best": float("inf"), "best_state": None, "hist": [], "run": 0.0, "n": 0}
    evals = [0.25, 0.5, 0.75, 1.0]
    log(f"[{exp['name']}] {len(tr)} training pairs, {len(va)} tune pairs | {planned} optimizer steps planned, budget {exp['budget_min']} min"
        + (f" | KD alpha {alpha} T {T}" if teacher is not None else ""))

    def evaluate():
        nll = pairs_nll(kind, model, va, data, cap, device, exp["infer_bs"], P)
        st["hist"].append({"progress": round(progress, 3), "step": step, "train_loss": round(st["run"] / max(1, st["n"]), 4),
                           "tune_nll": round(nll, 4), "minutes": round((time.time() - t0) / 60, 1)})
        h = st["hist"][-1]
        log(f"[{exp['name']}] progress {progress:.2f} step {step} | train loss {h['train_loss']:.4f} | tune NLL/token {nll:.4f} | {h['minutes']} min")
        st["run"], st["n"] = 0.0, 0
        if nll < st["best"]:
            st["best"] = nll
            st["best_state"] = {k: v.detach().cpu().clone() for k, v in trainable_state(model, tuner).items()}

    done = False
    for epoch in range(exp["epochs"]):
        for b in batches(tr, mb, rng):
            model.train()
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_mult(progress)
            part = [tr[i] for i in b]
            prompts, tgts = [make_prompt(p["seg"], p["cue"], cap) for p in part], [p["tgt"] for p in part]
            lg, y = target_logits(kind, model, prompts, tgts, data["pad"], data["start"], device, P)
            loss = F.cross_entropy(lg, y)
            if teacher is not None and alpha > 0:
                with torch.no_grad():
                    tl, _ = target_logits("D", teacher, prompts, tgts, data["pad"], data["start"], device)
                kd = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(tl / T, -1), reduction="batchmean") * T * T
                loss = (1 - alpha) * loss + alpha * kd
            scaler.scale(loss / accum).backward()
            st["run"], st["n"], micro = st["run"] + float(loss.detach()), st["n"] + 1, micro + 1
            if micro % accum:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            progress = min(1.0, max(step / planned, (time.time() - t0) / budget))
            while evals and progress >= evals[0]:
                evals.pop(0)
                evaluate()
            if progress >= 1.0:
                done = True
                break
        if done:
            break
    if evals or st["best_state"] is None:
        progress = 1.0
        evaluate()
    load_trainable(model, tuner, st["best_state"])
    return {"train_minutes": round((time.time() - t0) / 60, 1), "steps": step, "history": st["hist"], "best_tune_nll": st["best"],
            "peak_mem_train_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0}


def trainable_state(model, tuner):
    s = {f"m.{n}": p for n, p in model.named_parameters() if p.requires_grad}
    if tuner is not None:
        s["prefix"] = tuner.prefix
    return s


def load_trainable(model, tuner, state):
    named = dict(model.named_parameters())
    with torch.no_grad():
        for k, v in state.items():
            if k == "prefix":
                tuner.prefix.copy_(v)
            else:
                named[k[2:]].copy_(v)


# ---------------------------------------------------------------------------
# Writing summaries
# ---------------------------------------------------------------------------
def uniform_plan(plan):
    """Ablation: the same total length spread evenly over all segments (no positional profile)."""
    segs = [dict(s) for s in plan["segments"]]
    each = plan["planned_words"] / len(segs)
    for s in segs:
        s["budget"] = min(each, TGT_MAX_TOK / 1.7)
    keep = [k for k, s in enumerate(segs) if s["budget"] >= MIN_WORDS] or list(range(min(len(segs), 3)))
    return dict(plan, segments=segs, write=keep)


def write_summaries(kind, model, plans, data, scale, device, decode, bs, n_prefix=0):
    """-> (summaries, parts) where parts[i] = {segment index: text}."""
    cap, new_cap = limits(kind, model)
    cap -= n_prefix
    items, prompts, max_new = [], [], []
    for i, pl in enumerate(plans):
        for k in pl["write"]:
            s = pl["segments"][k]
            n = cue_n(s["budget"] * scale)
            items.append((i, k))
            prompts.append(make_prompt(s["ids"], data["cue"][(s["pos"], n)], cap))
            max_new.append(int(min(new_cap, s["budget"] * scale * 1.7 + 16)))
    outs = generate(kind, model, prompts, max_new, data["eos"], data["pad"], data["start"], device, bs, n_prefix,
                    vocab_limit=data.get("vocab"), banned=data.get("banned", ()))
    parts = [dict() for _ in plans]
    for (i, k), g in zip(items, outs):
        parts[i][k] = " ".join(decode(g).split())
    summaries = []
    for i, pl in enumerate(plans):
        txt = " ".join(parts[i][k] for k in sorted(parts[i]) if parts[i][k])
        summaries.append(txt or " ".join(pl["segments"][0]["text"].split()[:40]) or "The appeal was considered.")
    return summaries, parts


def run_experiment(exp, data, make_backbone, decode, device, log, out_dir, teacher_dir=None):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    model, d, kind = make_backbone(exp["model"])
    n_backbone = sum(p.numel() for p in model.parameters())
    n_tr, tuner = apply_method(model, kind, exp["method"], dict(exp, d_model=d))
    model.to(device)
    if tuner is not None:                           # the prefix lives outside the model: move it too
        tuner.to(device)
    P = tuner.P if tuner is not None else 0
    if tuner is not None:
        cap, _ = limits(kind, model)
        sample = sorted((make_prompt(p["seg"], p["cue"], cap - P) for p in data["pairs"]["train"][:300]), key=len)[-8:]
        init_prefix(kind, model, tuner, sample, data["pad"], device)
    teacher = None
    if teacher_dir is not None:                    # word-level KD from the LoRMA teacher (S1)
        tm, _, tk = make_backbone("D")
        apply_method(tm, tk, "lorma", exp["teacher"])
        state = torch.load(Path(teacher_dir) / "trainable.pt", map_location="cpu", weights_only=False)
        load_trainable(tm, None, state)
        teacher = tm.to(device).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        exp["teacher_params"] = sum(p.numel() for p in teacher.parameters())
    sto = storage_mb(model)
    log(f"[{exp['name']}] {exp['title']} | backbone {exp['model']} {n_backbone / 1e6:.1f}M ({sto} MB stored) | trainable {n_tr:,} "
        f"({100 * n_tr / n_backbone:.2f}%) | context {limits(kind, model)}")
    info = {"exp": exp, "n_backbone": n_backbone, "n_trainable": n_tr, "trainable_pct": round(100 * n_tr / n_backbone, 3),
            "storage_mb": sto}
    if exp["method"] == "qlora":
        errs = [m._err for m in model.modules() if isinstance(m, QLoRALinear)]
        info["nf4_relative_weight_error"] = round(float(np.mean(errs)), 4)
    info.update(train(exp, data, model, kind, tuner, device, log, teacher))
    if teacher is not None:
        del teacher, tm
        torch.cuda.empty_cache() if device.type == "cuda" else None
    if exp.get("save_trainable"):
        torch.save({k: v.detach().cpu() for k, v in trainable_state(model, tuner).items()}, out_dir / "trainable.pt")
    bs = exp["gen_bs"]
    t0 = time.time()
    tune = data["docs"]["tune"]
    scales = {}
    for sc in LEN_SCALES:
        summ, _ = write_summaries(kind, model, tune, data, sc, device, decode, bs, P)
        m, _ = evaluate(summ, tune, device)
        scales[str(sc)] = m
        log(f"[{exp['name']}]   tune, length x{sc}: BERTScore {m['bertscore_f1']} | R-1 {m['rouge1']:.4f} | R-L {m['rougeL']:.4f} | "
            f"{m['words']:.0f} words (refs {m['ref_words']:.0f})")
    best = max(LEN_SCALES, key=lambda s: primary(scales[str(s)]))
    t1 = time.time()
    dev = data["docs"]["dev"]
    dev_summ, dev_parts = write_summaries(kind, model, dev, data, best, device, decode, bs, P)
    gen_s = time.time() - t1
    dm, dev_bs = evaluate(dev_summ, dev, device)
    info.update({"length_scales_tune": scales, "chosen_scale": best, "dev": dm,
                 "dev_docs_per_min": round(len(dev) / max(1e-6, gen_s / 60), 2)})
    log(f"[{exp['name']}] dev (length x{best}): BERTScore {dm['bertscore_f1']} | R-1 {dm['rouge1']:.4f} | R-L {dm['rougeL']:.4f} | "
        f"{dm['words']:.0f} words | repeated 3-grams {dm['repeated_trigrams']:.3f} | novel 4-grams {dm['novel_4grams']:.3f} | "
        f"{info['dev_docs_per_min']} docs/min")
    if exp.get("ablate_plan"):
        us, _ = write_summaries(kind, model, [uniform_plan(p) for p in dev], data, best, device, decode, bs, P)
        um, _ = evaluate(us, dev, device)
        info["dev_uniform_budget"] = um
        log(f"[{exp['name']}] ablation, uniform budget per segment: BERTScore {um['bertscore_f1']} | R-1 {um['rouge1']:.4f} | R-L {um['rougeL']:.4f}")
    test_summ, test_parts = write_summaries(kind, model, data["docs"]["test"], data, best, device, decode, bs, P)
    info["generation_minutes"] = round((time.time() - t0) / 60, 1)
    rows = [{"id": pl["raw_id"], "summary": s} for pl, s in zip(data["docs"]["test"], test_summ)]
    write_submission(out_dir, rows)
    json.dump({"dev": {pl["id"]: {str(k): v for k, v in p.items()} for pl, p in zip(dev, dev_parts)},
               "test": {pl["id"]: {str(k): v for k, v in p.items()} for pl, p in zip(data["docs"]["test"], test_parts)}},
              open(out_dir / "parts.json", "w"))
    json.dump([{"id": pl["id"], "bertscore_f1": None if dev_bs is None else dev_bs[i], "summary": dev_summ[i], "reference": pl["summary"]}
               for i, pl in enumerate(dev)], open(out_dir / "dev_summaries.json", "w"), indent=1)
    json.dump(info, open(out_dir / "metrics.json", "w"), indent=1, default=str)
    log(f"[{exp['name']}] DONE -> {out_dir / 'submission.zip'} ({len(rows)} test summaries, {np.mean([len(r['summary'].split()) for r in rows]):.0f} words avg)")
    return info


def write_submission(out_dir, rows):
    import zipfile
    out_dir = Path(out_dir)
    seen = set()
    for r in rows:
        assert r["id"] not in seen and isinstance(r["summary"], str) and r["summary"].strip(), r["id"]
        seen.add(r["id"])
    with open(out_dir / "prediction.json", "w") as f:
        json.dump(rows, f)
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")


# ---------------------------------------------------------------------------
# System combination and extractive references (no GPU model)
# ---------------------------------------------------------------------------
def consensus(parts_by_run, plans):
    """Segment-level minimum-Bayes-risk combination: for every planned part, keep the candidate that
    agrees most (mean ROUGE-1 F1) with the other systems' versions of the same part."""
    summaries, picks = [], Counter()
    runs = list(parts_by_run)
    for pl in plans:
        out = []
        for k in pl["write"]:
            cands = [(r, parts_by_run[r][pl["id"]].get(str(k), "")) for r in runs]
            cands = [(r, c) for r, c in cands if c]
            if not cands:
                continue
            if len(cands) <= 2:
                r, c = cands[0]
            else:
                r, c = max(cands, key=lambda rc: np.mean([rouge(rc[1], o)[0] for r2, o in cands if r2 != rc[0]]))
            picks[r] += 1
            out.append(c)
        summaries.append(" ".join(out) or " ".join(pl["segments"][0]["text"].split()[:40]) or "The appeal was considered.")
    return summaries, dict(picks)


def extractive(plan, lead=False):
    """References without any language model: lead-N words, or per planned part the most central
    sentences of the segment (cosine to the segment's word centroid) up to the budget, in order."""
    if lead:
        words = " ".join(s["text"] for s in plan["segments"]).split()
        return " ".join(words[:int(plan["planned_words"])])
    out = []
    for k in plan["write"]:
        s = plan["segments"][k]
        sents = split_sentences(s["text"])
        cen = Counter(content_words(s["text"]))
        norm = math.sqrt(sum(v * v for v in cen.values())) or 1

        def score(x):
            c = Counter(content_words(x))
            return sum(c[w] * cen[w] for w in c) / (norm * (math.sqrt(sum(v * v for v in c.values())) or 1))
        ranked = sorted(range(len(sents)), key=lambda i: -score(sents[i]))
        pick, n = set(), 0
        for i in ranked:
            if n >= s["budget"]:
                break
            pick.add(i)
            n += len(sents[i].split())
        out.append(" ".join(sents[i] for i in sorted(pick)))
    return " ".join(out)


# ---------------------------------------------------------------------------
# Self-tests (tiny models, CPU, seconds)
# ---------------------------------------------------------------------------
def self_test(make_tiny, device, kinds=("D", "C", "A", "B")):
    import itertools
    torch.manual_seed(0)
    rng = np.random.RandomState(0)
    pad, start, eos = 0, 1, 2
    prompts = [list(rng.randint(3, 60, 9)), list(rng.randint(3, 60, 5))]
    tgts = [list(rng.randint(3, 60, 4)) + [eos], list(rng.randint(3, 60, 6)) + [eos]]
    for kind in kinds:
        m0, d = make_tiny(kind)
        base, y0 = target_logits(kind, m0.eval(), prompts, tgts, pad, start, device)
        # batched == one at a time (padding is invisible), teacher-forced
        one = torch.cat([target_logits(kind, m0, [p], [t], pad, start, device)[0] for p, t in zip(prompts, tgts)])
        assert (one - base).abs().max() < 1e-3, f"{kind}: padding changes the logits"
        for method in ("lorma", "lora", "adapter", "qlora", "full") + (("prefix",) if family(kind) == "D" else ()):
            m, _ = make_tiny(kind)
            m.load_state_dict(m0.state_dict())
            n_tr, tuner = apply_method(m, kind, method, {"r": 4, "alpha": 8, "bottleneck": 8, "n_prefix": 2, "d_model": d})
            P = tuner.P if tuner is not None else 0
            lg, _ = target_logits(kind, m.eval(), prompts, tgts, pad, start, device, P)
            if method in ("lorma", "lora", "adapter", "full"):
                assert (lg - base).abs().max() < 1e-3, f"{kind}/{method} does not start as the Phase-1 model"
            elif method == "qlora":
                cos = F.cosine_similarity(lg, base, -1).min()
                assert cos > 0.9, f"{kind}/qlora too far from the Phase-1 model ({cos:.3f})"
            else:                                   # prefix: only the prefix learns
                m.train()
                lg, y = target_logits(kind, m, prompts, tgts, pad, start, device, P)
                F.cross_entropy(lg, y).backward()
                assert tuner.prefix.grad is not None and tuner.prefix.grad.abs().sum() > 0
                assert all(p.grad is None for p in m.parameters())
                tuner.remove()
            if method in ("lorma", "lora", "qlora"):
                assert n_tr > 0 and all(n.split(".")[-1] in ("A", "B") for n, p in m.named_parameters() if p.requires_grad)
        # greedy generation with the KV cache == argmax of the full teacher-forced pass, batched == alone
        gp = [list(rng.randint(3, 60, 7)), list(rng.randint(3, 60, 7 if family(kind) == "AB" else 4))]
        g = generate(kind, m0, gp, [6, 6], eos=10 ** 6, pad=pad, start_id=start, device=device, bs=2, no_repeat=0, rep_penalty=1.0)
        lg, _ = target_logits(kind, m0, gp, [x + [3] for x in g], pad, start, device)
        am = lg.argmax(-1).tolist()
        k = 0
        for x in g:
            assert am[k:k + len(x)] == x, f"{kind}: cached generation differs from the full forward pass"
            k += len(x) + 1
        g1 = [generate(kind, m0, [p], [6], 10 ** 6, pad, start, device, 1, no_repeat=0, rep_penalty=1.0)[0] for p in gp]
        assert g1 == g, f"{kind}: batched generation differs from one-at-a-time"
        e = g[0][2]
        g2 = generate(kind, m0, gp, [6, 3], eos=e, pad=pad, start_id=start, device=device, bs=2, no_repeat=0, rep_penalty=1.0)
        assert g2[0] == g[0][:g[0].index(e)] and len(g2[1]) <= 3, f"{kind}: EOS / length stop"
    # alignment DP == brute force on small cases
    for _ in range(20):
        n, K = rng.randint(1, 5), rng.randint(1, 4)
        segs = [" ".join(f"w{rng.randint(0, 12)}" for _ in range(8)) for _ in range(K)]
        sents = [" ".join(f"w{rng.randint(0, 12)}" for _ in range(4)) for _ in range(n)]
        a, sc = align(sents, segs, {})
        S = np.array([[len(set(content_words(s)) & set(content_words(g))) / max(1, len(set(content_words(s)))) for g in segs] for s in sents])

        def val(asg):
            return sum(S[j, k] for j, k in enumerate(asg)) - 0.1 * sum(asg[j] < asg[j - 1] for j in range(1, len(asg)))
        bf = max(val(x) for x in itertools.product(range(K), repeat=n))
        assert abs(val(a) - bf) < 1e-9, "alignment is not optimal"
    # plan: budgets add up, parts below MIN_WORDS skipped
    d = {"id": "x", "raw_id": "x", "sents": None, "summary": "s", "text": " ".join(["Facts here."] * 600)}
    pl = plan_doc(d, {"a": math.log(0.2), "b": 1.0, "lo": 10, "hi": 1e6}, [0.3] * 10)
    tot = sum(pl["segments"][k]["budget"] for k in pl["write"])
    assert abs(tot - min(pl["planned_words"], len(pl["write"]) * TGT_MAX_TOK / 1.7)) < 1 and pl["write"]
    pe = plan_doc(dict(d, text=""), {"a": math.log(0.2), "b": 1.0, "lo": 10, "hi": 1e6}, [0.3] * 10)
    assert pe["write"] and all(np.isfinite(s["budget"]) for s in pe["segments"]), "empty document must still plan"
    pr = plan_doc(dict(d, text=" ".join(["Facts here."] * 20000)), {"a": math.log(800), "b": 0.0, "lo": 10, "hi": 1e6},
                  [0.5, 0, 0, 0, 0, 0, 0, 0, 0, 0.5])
    assert abs(sum(pr["segments"][k]["budget"] for k in pr["write"]) - 800) < 1, "skipped parts' words must be re-shared"
    assert rouge("the court held", "the court held") == (1.0, 1.0)
    s, picks = consensus({"r1": {"x": {"0": "a b c d"}}, "r2": {"x": {"0": "a b c e"}}, "r3": {"x": {"0": "z y"}}},
                         [{"id": "x", "write": [0], "segments": [{"text": "q"}]}])
    assert s[0] in ("a b c d", "a b c e"), s
    print(f"sum_lib self-test passed: LoRMA / LoRA / adapters / full start as the Phase-1 model, QLoRA as its 4-bit copy, prefix "
          f"trains only the prefix; padding-safe; cached greedy generation == full forward pass, batched == alone, EOS / length "
          f"stops ({', '.join(kinds)}); alignment DP optimal; plan budgets; ROUGE; consensus.")

In [ ]:
%%writefile /kaggle/working/sum_train.py
"""sum_train.py -- one Phase-6 experiment: python sum_train.py <config.json> <experiment> <gpu index or -1>"""
import importlib
import json
import sys
from pathlib import Path

import torch

cfg = json.load(open(sys.argv[1]))
name, gpu = sys.argv[2], int(sys.argv[3])
sys.path.insert(0, cfg["work_dir"])
import sum_lib as SL  # noqa: E402

device = torch.device(f"cuda:{gpu}") if gpu >= 0 and torch.cuda.is_available() else torch.device("cpu")
if device.type == "cuda":
    torch.cuda.set_device(device)
exp = dict(cfg["common"], **cfg["experiments"][name], name=name)
TOK = {"A": "T1", "B": "T1", "C": "T2", "D": "T2", "Ds": "T2"}[exp["model"]]
data = torch.load(cfg["data"][TOK], weights_only=False)

if TOK == "T2":
    import legal_d
    legal_d.load_tokenizer(cfg["tokenizer"]["T2"])
    decode = legal_d.sp.decode
else:
    from tokenizers import Tokenizer
    _t1 = Tokenizer.from_file(cfg["tokenizer"]["T1"])

    def decode(ids):
        return _t1.decode(list(ids), skip_special_tokens=True)


def make_backbone(kind):
    """Phase-1 model from its checkpoint -> (module, d_model, kind)."""
    if kind in ("D", "Ds"):
        import legal_d
        ck = torch.load(cfg["ckpt"]["D"], map_location="cpu", weights_only=False)
        m = legal_d.build_model(ck["config"])
        m.load_state_dict(ck["model_state"])
        if kind == "Ds":                          # distillation student: every other encoder AND decoder layer
            keep = exp["student_layers"]
            m.enc_blocks = torch.nn.ModuleList([m.enc_blocks[i] for i in keep if i < len(m.enc_blocks)])
            m.dec_blocks = torch.nn.ModuleList([m.dec_blocks[i] for i in keep if i < len(m.dec_blocks)])
        m.grad_ckpt = exp.get("grad_ckpt", True)
        return m, ck["config"]["d_model"], kind
    mod = importlib.import_module({"A": "a_model", "B": "b_model", "C": "c_model"}[kind])
    ck = torch.load(cfg["ckpt"][kind], map_location="cpu", weights_only=False)
    m = mod.build_model(ck["config"])
    m.load_state_dict(ck["model_state"])
    mod.USE_GRAD_CHECKPOINTING = exp.get("grad_ckpt", True)
    return m, ck["config"]["d_model"], kind


def log(*a):
    print(*a, flush=True)


teacher_dir = Path(cfg["out_dir"]) / exp["teacher_run"] if exp.get("teacher_run") else None
SL.run_experiment(exp, data, make_backbone, decode, device, log, Path(cfg["out_dir"]) / name, teacher_dir)

## 3. Data: download, clean, hold out tune / dev, align, learn the plan

In [ ]:
sys.path.insert(0, str(WORK_DIR))
import sum_lib as SL  # noqa: E402
from huggingface_hub import hf_hub_download  # noqa: E402
from kaggle_secrets import UserSecretsClient  # noqa: E402

HF_TOKEN = UserSecretsClient().get_secret("HF")
REPO = "Exploration-Lab/CS-781-Capstone"


def hf_file(*names):
    # first of the candidate paths that exists in the repo (the capstone doc names the first one)
    last = None
    for n in names:
        try:
            return hf_hub_download(REPO, n, repo_type="dataset", token=HF_TOKEN)
        except Exception as e:      # noqa: BLE001
            last = e
    try:
        from huggingface_hub import list_repo_files
        have = [f for f in list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN) if "6" in f.split("/")[0]]
    except Exception:               # noqa: BLE001
        have = "?"
    raise FileNotFoundError(f"none of {names} in {REPO} ({last}). Phase-6 files in the repo: {have}")


t0 = time.time()
RAW = {"train": hf_file("Phase_6_Summ/Train/train.json", "Phase_6_Summ/Train/train.jsonl"),
       "test": hf_file("Phase_6_Summ/Test/test.json", "Phase_6_Summ/Test/test.jsonl")}
raw = {s: SL.parse_docs(SL.load_json_any(p)) for s, p in RAW.items()}
print(f"downloaded + parsed in {time.time() - t0:.0f}s:", {s: len(v) for s, v in raw.items()})
first = SL.load_json_any(RAW["train"])
print("train fields:", list((first[0] if isinstance(first, list) else next(iter(first.values()))).keys()))
for s in raw:
    empty = sum(not d["text"] for d in raw[s])
    assert empty <= 0.05 * len(raw[s]), f"{s}: {empty} documents have no text -- check the field names"
assert sum(d["summary"] is not None for d in raw["train"]) > 0.9 * len(raw["train"]), "train needs a 'summary' field"
SPLIT, CLEAN = SL.clean_and_split(raw["train"], raw["test"])
print("cleaning + split:", CLEAN)
del raw, first

# ---- align parts of the judgment with parts of the summary; learn the plan --------------------------
t0 = time.time()
IDF = SL.build_idf(SPLIT["train"])
PAIRS, STATS = SL.make_pairs(SPLIT["train"], IDF)
TUNE_PAIRS, _ = SL.make_pairs(SPLIT["tune"], IDF)
LM = SL.fit_length_model(STATS["lengths"])
PROFILE = STATS["profile"]
PLANS = {s: [SL.plan_doc(d, LM, PROFILE) for d in SPLIT[s]] for s in ("tune", "dev", "test")}
dw = np.array([d for d, _ in STATS["lengths"]])
sw = np.array([s for _, s in STATS["lengths"]])
sc = np.array(STATS["align_scores"])
REPORT = {"cleaning": CLEAN, "length_model": LM, "profile_words_per_doc_word_by_decile": [round(x, 4) for x in PROFILE]}
print(f"aligned in {time.time() - t0:.0f}s")
print(f"judgment words: median {np.median(dw):.0f}, 90th pct {np.quantile(dw, 0.9):.0f}, max {dw.max()} | summary words: median "
      f"{np.median(sw):.0f}, 90th pct {np.quantile(sw, 0.9):.0f} | compression: median {np.median(sw / np.maximum(dw, 1)):.3f}")
print(f"length model: summary words = {np.exp(LM['a']):.2f} x judgment_words^{LM['b']:.2f}, clipped to [{LM['lo']:.0f}, {LM['hi']:.0f}]")
print("where summaries draw from (summary words per judgment word, by position decile):", [round(x, 3) for x in PROFILE])
print(f"alignment: {len(sc)} summary sentences | median support {np.median(sc):.2f} | {100 * np.mean(sc < 0.2):.1f}% weakly "
      f"supported (< 0.2: abstract / paraphrased content)")
nseg = np.array([len(p["segments"]) for p in PLANS["dev"]])
nw = np.array([len(p["write"]) for p in PLANS["dev"]])
print(f"pairs: {len(PAIRS)} train, {len(TUNE_PAIRS)} tune | sub-summary words: median {np.median([p['n'] for p in PAIRS]):.0f} | "
      f"dev: {nseg.mean():.1f} parts per judgment, {nw.mean():.1f} written ({100 * (1 - nw.sum() / nseg.sum()):.0f}% skipped by the plan)")
print("planned vs reference words (dev):", round(float(np.mean([p["planned_words"] for p in PLANS["dev"]]))),
      "vs", round(float(np.mean([len(p["summary"].split()) for p in PLANS["dev"]]))))
REPORT.update({"judgment_words": {"median": float(np.median(dw)), "p90": float(np.quantile(dw, 0.9))},
               "summary_words": {"median": float(np.median(sw)), "p90": float(np.quantile(sw, 0.9))},
               "align_support_median": float(np.median(sc)), "align_weak_share": float(np.mean(sc < 0.2)),
               "pairs_train": len(PAIRS), "parts_per_dev_doc": float(nseg.mean()), "written_per_dev_doc": float(nw.mean())})
torch.save(PLANS, OUT_DIR / "p6_plans.pt")

## 4. Tokenize (T2 for C and D, T1 for A and B)

In [ ]:
import legal_d  # noqa: E402
from tokenizers import Tokenizer  # noqa: E402

legal_d.load_tokenizer(T2_PATH)
sp = legal_d.sp
t1 = Tokenizer.from_file(str(T1_PATH))


def enc_t2(texts):
    try:
        return sp.encode(texts, out_type=int, num_threads=os.cpu_count() or 4)
    except TypeError:
        return sp.encode(texts, out_type=int)


def enc_t1(texts):
    return [e.ids for e in t1.encode_batch(texts)]


SPEC = {"T2": (enc_t2, sp.pad_id() if sp.pad_id() >= 0 else 0, legal_d.EOS_ID, legal_d.DEC_START_ID),
        "T1": (enc_t1, t1.token_to_id("<pad>") or 0, t1.token_to_id("<eos>") if t1.token_to_id("<eos>") is not None else 0,
               t1.token_to_id("<eos>") or 0)}
U16 = lambda x: np.asarray(x, dtype=np.uint16)
DATA_PATH = {}
for name, (enc, pad, eos, start) in SPEC.items():
    t0 = time.time()
    cue_keys = [(pos, n) for pos in SL.POS_WORDS for n in SL.CUE_NS]
    cue = dict(zip(cue_keys, enc([SL.cue_text(p, n) for p, n in cue_keys])))

    def pairs_of(P):
        segs = enc([p["seg"] for p in P])
        tg = enc([" " + p["tgt"] for p in P])
        return [{"seg": U16(s), "cue": cue[(p["pos"], SL.cue_n(p["n"]))], "tgt": U16(list(t[:SL.TGT_MAX_TOK - 1]) + [eos])}
                for p, s, t in zip(P, segs, tg)]
    docs = {}
    for s, plans in PLANS.items():
        flat = [seg["text"] for pl in plans for seg in pl["segments"]]
        ids = iter(enc(flat))
        docs[s] = [dict(pl, segments=[dict(seg, ids=U16(next(ids))) for seg in pl["segments"]]) for pl in plans]
    vocab = sp.get_piece_size() if name == "T2" else t1.get_vocab_size()
    special = {sp.pad_id(), sp.unk_id(), sp.bos_id()} if name == "T2" else {t1.token_to_id(x) for x in ("<pad>", "<unk>", "<bos>", "<s>")}
    banned = sorted(i for i in special - {eos} if i is not None and 0 <= i < vocab)
    data = {"tokenizer": name, "pad": pad, "eos": eos, "start": start, "cue": cue, "vocab": vocab, "banned": banned,
            "pairs": {"train": pairs_of(PAIRS), "tune": pairs_of(TUNE_PAIRS)}, "docs": docs}
    assert max(int(p["seg"].max(initial=0)) for p in data["pairs"]["train"]) < 65536
    seg_tok = np.array([len(p["seg"]) for p in data["pairs"]["train"]])
    tgt_tok = np.array([len(p["tgt"]) for p in data["pairs"]["train"]])
    print(f"{name}: tokenized in {time.time() - t0:.0f}s | part tokens: median {np.median(seg_tok):.0f}, 90th pct "
          f"{np.quantile(seg_tok, 0.9):.0f} | sub-summary tokens: median {np.median(tgt_tok):.0f}, cut at {SL.TGT_MAX_TOK}: "
          f"{100 * np.mean(tgt_tok >= SL.TGT_MAX_TOK):.1f}%")
    REPORT[f"{name}_tokens"] = {"part_median": float(np.median(seg_tok)), "sub_summary_median": float(np.median(tgt_tok))}
    DATA_PATH[name] = str(OUT_DIR / f"p6_data_{name}.pt")
    torch.save(data, DATA_PATH[name])
    del data, docs
json.dump(REPORT, open(OUT_DIR / "data_report.json", "w"), indent=1, default=str)
del PAIRS, TUNE_PAIRS
gc.collect()
# fetch roberta-large once (local scoring only) so the runs do not download it in parallel
print("BERTScore check:", SL.bertscore(["the court allowed the appeal"], ["the appeal was allowed by the court"]))
SL._SCORER.clear()                                  # free GPU 0 for the runs
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 5. Self-tests (tiny models, seconds)

In [ ]:
import a_model  # noqa: E402
import b_model  # noqa: E402
import c_model  # noqa: E402


def make_tiny(kind):
    if kind == "D":
        cfg = dict(legal_d.MODEL_D_CONFIG, vocab_size=64, d_model=32, n_enc_layer=2, n_dec_layer=2, n_head=4,
                   n_kv_head=2, d_head=8, d_ff=48, max_enc_len=64, max_dec_len=64)
        m = legal_d.build_model(cfg).eval()
        return m, 32
    if kind == "C":
        cfg = dict(vocab_size=64, d_model=32, n_layer=2, n_head=4, n_kv_head=2, d_head=8, d_ff=48, max_seq_len=320,
                   qk_norm=True, bias=True, mlp="gelu", tokenizer_id="T2", attn_type="prefix", tied_embeddings=True)
        return c_model.build_model(cfg).eval(), 32
    if kind == "A":
        cfg = dict(vocab_size=64, d_model=32, n_layer=2, n_head=4, n_kv_head=2, d_head=8, d_ff=48, max_seq_len=320)
        return a_model.build_model(cfg).eval(), 32
    if kind == "B":
        cfg = dict(vocab_size=64, d_model=32, n_layer=3, n_head=4, n_kv_head=1, d_head=8, d_ff=48, max_seq_len=320,
                   window=4, global_every=3, qk_norm=True, tokenizer_id="T1", attn_type="local+global", tied_embeddings=True)
        return b_model.build_model(cfg).eval(), 32
    raise KeyError(kind)


for _m in (a_model, b_model, c_model):
    _m.USE_GRAD_CHECKPOINTING = True
SL.self_test(make_tiny, torch.device("cpu"), kinds=("D", "C", "A", "B"))

## 6. Experiment settings

`budget_min` caps training; `est_min` (training + writing + scoring) is only used to skip a run that could not finish in the session.

In [ ]:
COMMON = dict(epochs=2, seed=13, val_pairs=400, grad_ckpt=True)
EXPERIMENTS = {
    "S1": dict(title="D + LoRMA", model="D", method="lorma", r=16, alpha=32, lr=5e-4, micro_bs=8, accum=2,
               budget_min=55, infer_bs=16, gen_bs=32, save_trainable=True, ablate_plan=True, est_min=100, submit=True),
    "S2": dict(title="C + Houlsby adapters", model="C", method="adapter", bottleneck=64, lr=1e-3, micro_bs=8, accum=2,
               budget_min=55, infer_bs=16, gen_bs=24, est_min=100, submit=True),
    "S6": dict(title="KD student: half of D <- S1", model="Ds", method="full", student_layers=[0, 2, 4, 6, 8, 10], lr=1e-4,
               micro_bs=8, accum=2, budget_min=55, infer_bs=16, gen_bs=32, teacher_run="S1", teacher=dict(r=16, alpha=32),
               kd_alpha=0.5, kd_T=2.0, est_min=110, submit=True),
    "S3": dict(title="B + QLoRA (NF4 4-bit + LoRA)", model="B", method="qlora", r=16, alpha=32, lr=5e-4, micro_bs=6, accum=3,
               budget_min=60, infer_bs=12, gen_bs=24, est_min=115, submit=True),
    "S4": dict(title="A + LoRA", model="A", method="lora", r=16, alpha=32, lr=5e-4, micro_bs=4, accum=4,
               budget_min=60, infer_bs=8, gen_bs=16, est_min=125, submit=True),
    "S5": dict(title="D + prefix tuning (encoder, 32 states/layer)", model="D", method="prefix", n_prefix=32, lr=3e-3,
               micro_bs=8, accum=2, budget_min=50, infer_bs=16, gen_bs=32, est_min=95, submit=True),
}
RUN_ORDER = [("S1", []), ("S2", []), ("S6", ["S1"]), ("S3", []), ("S4", []), ("S5", [])]
TIME_LIMIT_MIN = 11 * 60          # Kaggle stops at 12 h: never start a run that would end after 11 h
CFG = {"work_dir": str(WORK_DIR), "data": DATA_PATH, "ckpt": CKPT, "out_dir": str(OUT_DIR), "common": COMMON,
       "experiments": EXPERIMENTS, "tokenizer": {"T1": str(T1_PATH), "T2": str(T2_PATH)}}
CONFIG_PATH = WORK_DIR / "sum_config.json"
json.dump(CFG, open(CONFIG_PATH, "w"), indent=1)
for name, e in EXPERIMENTS.items():
    print(f"{name}: {e['title']}")

## 7. Run (two at a time, one per GPU; S6 waits for its teacher S1)

In [ ]:
def stream(proc, tag):
    with open(OUT_DIR / f"log_{tag}.txt", "w") as f:
        for line in proc.stdout:
            f.write(line)
            print(line if line.startswith(f"[{tag}]") else f"[{tag}] {line}", end="", flush=True)


queue, running, codes = list(RUN_ORDER), {}, {}
slots = list(range(N_GPUS))[:2] if N_GPUS else [-1]
while queue or running:
    for slot in slots:
        if slot in running:
            continue
        ready = [q for q in queue if all(d in codes for d in q[1])]
        if not ready:
            continue
        name, deps = ready[0]
        queue.remove(ready[0])
        if any(codes.get(d) != 0 for d in deps):
            print(f">>> skipping {name}: {deps} did not all finish", flush=True)
            codes[name] = -1
            continue
        used = (time.time() - NB_T0) / 60
        if used + EXPERIMENTS[name]["est_min"] > TIME_LIMIT_MIN:
            print(f">>> skipping {name}: {used:.0f} min used, it needs ~{EXPERIMENTS[name]['est_min']} more", flush=True)
            codes[name] = -2
            continue
        env = dict(os.environ, PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
        proc = subprocess.Popen([sys.executable, str(WORK_DIR / "sum_train.py"), str(CONFIG_PATH), name, str(slot)],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        th = threading.Thread(target=stream, args=(proc, name), daemon=True)
        th.start()
        running[slot] = (name, proc, th)
        print(f">>> started {name} ({EXPERIMENTS[name]['title']}) on {'GPU ' + str(slot) if slot >= 0 else 'CPU'} "
              f"({(time.time() - NB_T0) / 60:.0f} min into the notebook)", flush=True)
    for slot, (name, proc, th) in list(running.items()):
        if proc.poll() is not None:
            th.join()
            codes[name] = proc.returncode
            print(f">>> {name} finished with code {proc.returncode} ({(time.time() - NB_T0) / 60:.0f} min into the notebook)", flush=True)
            del running[slot]
    time.sleep(5)
print("Exit codes:", codes)

## 8. S7 consensus + references without a language model

In [ ]:
# S7 (segment-level consensus of S1-S6) + references without any language model, scored on dev
PLANS = torch.load(OUT_DIR / "p6_plans.pt", weights_only=False)
members = [n for n, e in EXPERIMENTS.items() if e["submit"] and (OUT_DIR / n / "parts.json").exists()]
parts = {n: json.load(open(OUT_DIR / n / "parts.json")) for n in members}
EXTRA = {}
if len(members) >= 3:
    S7 = OUT_DIR / "S7"
    S7.mkdir(exist_ok=True)
    dev_s, dev_picks = SL.consensus({n: parts[n]["dev"] for n in members}, PLANS["dev"])
    m7, bs7 = SL.evaluate(dev_s, PLANS["dev"])
    test_s, test_picks = SL.consensus({n: parts[n]["test"] for n in members}, PLANS["test"])
    SL.write_submission(S7, [{"id": pl["raw_id"], "summary": s} for pl, s in zip(PLANS["test"], test_s)])
    json.dump({"exp": {"name": "S7", "title": "Consensus of " + " + ".join(members)}, "dev": m7, "dev_picks": dev_picks,
               "test_picks": test_picks}, open(S7 / "metrics.json", "w"), indent=1)
    json.dump([{"id": pl["id"], "bertscore_f1": None if bs7 is None else bs7[i], "summary": dev_s[i], "reference": pl["summary"]}
               for i, pl in enumerate(PLANS["dev"])], open(S7 / "dev_summaries.json", "w"), indent=1)
    print(f"S7 consensus of {members}: dev BERTScore {m7['bertscore_f1']} | R-1 {m7['rouge1']:.4f} | R-L {m7['rougeL']:.4f} | "
          f"parts taken from each run: {dev_picks}")
for name, lead in (("Lead (first N words)", True), ("Extract (central sentences per planned part)", False)):
    s = [SL.extractive(pl, lead=lead) for pl in PLANS["dev"]]
    EXTRA[name], _ = SL.evaluate(s, PLANS["dev"])
    print(f"{name}: dev BERTScore {EXTRA[name]['bertscore_f1']} | R-1 {EXTRA[name]['rouge1']:.4f} | R-L {EXTRA[name]['rougeL']:.4f}")
json.dump(EXTRA, open(OUT_DIR / "references.json", "w"), indent=1)

## 9. Results, analysis tables, submissions

In [ ]:
rows, by = [], {}
for name in [n for n, _ in RUN_ORDER] + ["S7"]:
    mp = OUT_DIR / name / "metrics.json"
    if not mp.exists():
        print(f"{name}: no metrics (see its log above)")
        continue
    m = json.load(open(mp))
    e = EXPERIMENTS.get(name, {"title": m["exp"]["title"], "model": "-", "method": "consensus", "submit": True})
    d = m["dev"]
    r = {"run": name, "title": e["title"], "model": e["model"], "technique": e.get("method"), "trainable": m.get("n_trainable", 0),
         "trainable_pct": m.get("trainable_pct", 0), "storage_mb": m.get("storage_mb", 0), "length_scale": m.get("chosen_scale", "-"),
         "dev_bertscore": d["bertscore_f1"], "dev_rouge1": d["rouge1"], "dev_rougeL": d["rougeL"], "words": d["words"],
         "repeated_trigrams": d["repeated_trigrams"], "novel_4grams": d["novel_4grams"], "numbers_in_source": d["numbers_in_source"],
         "ref_coverage": d.get("ref_content_covered"), "train_min": m.get("train_minutes", 0), "gen_min": m.get("generation_minutes", 0),
         "peak_gb": m.get("peak_mem_train_gb", 0), "dev_docs_per_min": m.get("dev_docs_per_min", 0), "submit": e["submit"]}
    rows.append(r)
    by[name] = (r, m)
assert rows, "No run finished -- read the error lines above."
with open(OUT_DIR / "runs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)
print(f"{'run':4s} {'title':40s} {'trainable':>12s} {'%':>6s} {'MB':>5s} {'x':>5s} {'BERTSc':>7s} {'R-1':>6s} {'R-L':>6s} {'words':>6s} "
      f"{'rep3':>5s} {'nov4':>5s} {'num':>5s} {'cov':>5s} {'train':>6s} {'gen':>5s} {'doc/m':>6s}")
for r in rows:
    print(f"{r['run']:4s} {r['title'][:40]:40s} {r['trainable']:>12,} {r['trainable_pct']:6.2f} {r['storage_mb']:5.0f} {str(r['length_scale']):>5s} "
          f"{(r['dev_bertscore'] or 0):7.4f} {r['dev_rouge1']:6.4f} {r['dev_rougeL']:6.4f} {r['words']:6.0f} {r['repeated_trigrams']:5.3f} "
          f"{r['novel_4grams']:5.3f} {r['numbers_in_source']:5.3f} {(r['ref_coverage'] or 0):5.3f} {r['train_min']:6.1f} {r['gen_min']:5.1f} "
          f"{r['dev_docs_per_min']:6.1f}")
for k, v in EXTRA.items():
    print(f"     {k:40s} {'':>12s} {'':>6s} {'':>5s} {'':>5s} {(v['bertscore_f1'] or 0):7.4f} {v['rouge1']:6.4f} {v['rougeL']:6.4f} {v['words']:6.0f} "
          f"{v['repeated_trigrams']:5.3f} {v['novel_4grams']:5.3f}")
print("\nlength scale chosen on tune documents (BERTScore / R-1 / words):")
for name, (r, m) in by.items():
    if "length_scales_tune" in m:
        print(f"  {name}: " + " | ".join(f"x{s}: {v['bertscore_f1']} / {v['rouge1']:.3f} / {v['words']:.0f}" for s, v in m["length_scales_tune"].items())
              + f"  -> x{m['chosen_scale']}")
if "S1" in by and "dev_uniform_budget" in by["S1"][1]:
    u = by["S1"][1]["dev_uniform_budget"]
    print(f"\nlong-document plan (S1): learned positional budget BERTScore {by['S1'][0]['dev_bertscore']} R-1 {by['S1'][0]['dev_rouge1']:.4f} "
          f"vs uniform budget BERTScore {u['bertscore_f1']} R-1 {u['rouge1']:.4f}")
if "S6" in by and "S1" in by:
    s6, s1 = by["S6"][1], by["S1"][1]
    print(f"\ndistillation: teacher S1 {s1['n_backbone'] / 1e6:.1f}M, {s1['dev_docs_per_min']} dev docs/min, BERTScore {s1['dev']['bertscore_f1']} | "
          f"student S6 {s6['n_backbone'] / 1e6:.1f}M ({100 * s6['n_backbone'] / s1['n_backbone']:.0f}%), {s6['dev_docs_per_min']} docs/min, "
          f"BERTScore {s6['dev']['bertscore_f1']}")
if "S3" in by and by["S3"][1].get("nf4_relative_weight_error") is not None:
    print(f"QLoRA: B stored in {by['S3'][1]['storage_mb']} MB; 4-bit weights {100 * by['S3'][1]['nf4_relative_weight_error']:.1f}% from Phase-1 weights")
for name, (r, m) in by.items():
    if "history" in m:
        print(f"{name} training: " + ", ".join(f"{h['progress']}: {h['tune_nll']}" for h in m["history"]))
best = max((r for r in rows if r["dev_bertscore"] is not None), key=lambda r: r["dev_bertscore"], default=None)
if best:
    ex = json.load(open(OUT_DIR / best["run"] / "dev_summaries.json"))[0]
    print(f"\nexample ({best['run']}, dev {ex['id']}):\nSUMMARY: {ex['summary'][:700]}\nREFERENCE: {ex['reference'][:700]}")
with zipfile.ZipFile(OUT_DIR / "report_files.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT_DIR.rglob("*")):
        if f.is_file() and f.suffix in (".json", ".csv", ".txt", ".log") and f.name not in ("prediction.json", "parts.json"):
            z.write(f, arcname=str(f.relative_to(OUT_DIR)))
print(f"\nAll tables for the report: {OUT_DIR / 'report_files.zip'}")
print("\nSubmit these (Output tab), best first:")
for r in sorted(rows, key=lambda r: -(r["dev_bertscore"] or r["dev_rouge1"])):
    zf = OUT_DIR / r["run"] / "submission.zip"
    if zf.exists():
        pred = json.loads(zipfile.ZipFile(zf).read("prediction.json"))
        assert len(pred) == len(PLANS["test"]) and all(p["summary"].strip() for p in pred)
        print(f"  {r['run']} ({r['title']}): {zf}  [dev BERTScore {r['dev_bertscore']}, R-1 {r['dev_rouge1']:.4f}]")